In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/dimaspashaakrilian/dsc-itb/Data_Klaim.csv
/kaggle/input/datasets/dimaspashaakrilian/dsc-itb/sample_submission.csv
/kaggle/input/datasets/dimaspashaakrilian/dsc-itb/Data_Polis.csv


# Metric Lock

In [2]:
# =========================
# STAGE 1 — Metric Lock
# (Replicate Kaggle Score: avg(MAPE_freq, MAPE_sev, MAPE_total))
# =========================

import numpy as np
import pandas as pd

BASE_PATH = "/kaggle/input/datasets/dimaspashaakrilian/dsc-itb/"
PATH_KLAIM = BASE_PATH + "Data_Klaim.csv"
PATH_POLIS = BASE_PATH + "Data_Polis.csv"
PATH_SUB   = BASE_PATH + "sample_submission.csv"

# -------------------------
# Utils
# -------------------------
def clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
          .str.strip()
          .str.lower()
          .str.replace(r"\s+", "_", regex=True)
          .str.replace(r"[^a-z0-9_]+", "", regex=True)
    )
    return df

def mape(y_true, y_pred, eps=1e-9) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.maximum(np.abs(y_true), eps)
    return float(np.mean(np.abs((y_true - y_pred) / denom)))

def month_to_idprefix(ym: str) -> str:
    # ym: "YYYY-MM" -> "YYYY_MM"
    y, m = ym.split("-")
    return f"{y}_{m}"

def build_monthly_event(klaim_df: pd.DataFrame) -> pd.DataFrame:
    df = clean_columns(klaim_df)
    # kolom wajib (setelah clean)
    need = ["claim_id", "tanggal_pasien_masuk_rs", "nominal_klaim_yang_disetujui"]
    miss = [c for c in need if c not in df.columns]
    if miss:
        raise ValueError(f"Kolom wajib tidak ditemukan: {miss}. Cek nama kolom di CSV kamu.")

    df["tanggal_pasien_masuk_rs"] = pd.to_datetime(df["tanggal_pasien_masuk_rs"], errors="coerce")
    df = df.dropna(subset=["tanggal_pasien_masuk_rs"])

    df["year_month"] = df["tanggal_pasien_masuk_rs"].dt.to_period("M").astype(str)

    monthly = (
        df.groupby("year_month")
          .agg(
              frequency=("claim_id", "count"),
              total_claim=("nominal_klaim_yang_disetujui", "sum"),
          )
          .reset_index()
          .sort_values("year_month")
          .reset_index(drop=True)
    )
    monthly["severity"] = monthly["total_claim"] / monthly["frequency"].replace(0, np.nan)
    monthly["severity"] = monthly["severity"].fillna(0.0)
    return monthly

def build_truth_map(monthly: pd.DataFrame) -> dict:
    """
    Build map id -> y_true sesuai format sample_submission:
      YYYY_MM_Claim_Frequency
      YYYY_MM_Claim_Severity
      YYYY_MM_Total_Claim
    """
    truth = {}
    for _, r in monthly.iterrows():
        prefix = month_to_idprefix(r["year_month"])
        truth[f"{prefix}_Claim_Frequency"] = float(r["frequency"])
        truth[f"{prefix}_Claim_Severity"]  = float(r["severity"])
        truth[f"{prefix}_Total_Claim"]     = float(r["total_claim"])
    return truth

def score_pred_map(pred_map: dict, truth_map: dict, ids_to_score: list) -> dict:
    """
    Hitung MAPE per target + skor akhir seperti Kaggle.
    pred_map: dict {id: pred_value}
    truth_map: dict {id: true_value}
    ids_to_score: list id yang dinilai (mis. dari sample_submission)
    """
    # filter hanya id yang memang ada truth-nya (untuk backtest/historis)
    valid_ids = [i for i in ids_to_score if i in truth_map]
    if len(valid_ids) == 0:
        raise ValueError("Tidak ada id yang overlap antara ids_to_score dan truth_map. "
                         "Kalau backtest, pastikan bulan valid benar-benar ada di data historis.")

    def collect(suffix):
        ids = [i for i in valid_ids if i.endswith(suffix)]
        y_true = [truth_map[i] for i in ids]
        y_pred = [float(pred_map.get(i, np.nan)) for i in ids]
        if np.any(np.isnan(y_pred)):
            missing = [ids[k] for k,v in enumerate(y_pred) if np.isnan(v)]
            raise ValueError(f"Prediksi belum lengkap, id berikut belum ada di pred_map: {missing[:10]} (dst)")
        return y_true, y_pred, ids

    yt_f, yp_f, _ = collect("_Claim_Frequency")
    yt_s, yp_s, _ = collect("_Claim_Severity")
    yt_t, yp_t, _ = collect("_Total_Claim")

    m_f = mape(yt_f, yp_f)
    m_s = mape(yt_s, yp_s)
    m_t = mape(yt_t, yp_t)
    final = (m_f + m_s + m_t) / 3.0

    return {
        "MAPE_freq": m_f,
        "MAPE_sev":  m_s,
        "MAPE_total": m_t,
        "FINAL_SCORE": final,
        "n_freq": len(yt_f),
        "n_sev":  len(yt_s),
        "n_total": len(yt_t),
    }

def get_next_months(train_end_ym: str, horizon=5) -> list:
    """
    train_end_ym: "YYYY-MM"
    return list of next months in "YYYY-MM" length=horizon
    """
    p = pd.Period(train_end_ym, freq="M")
    return [(p + i).strftime("%Y-%m") for i in range(1, horizon+1)]

def ids_for_months(months_ym: list) -> list:
    ids = []
    for ym in months_ym:
        prefix = month_to_idprefix(ym)
        ids += [
            f"{prefix}_Claim_Frequency",
            f"{prefix}_Claim_Severity",
            f"{prefix}_Total_Claim",
        ]
    return ids

# -------------------------
# Load data
# -------------------------
klaim_raw = pd.read_csv(PATH_KLAIM)
polis_raw = pd.read_csv(PATH_POLIS)
sample_sub = pd.read_csv(PATH_SUB)

# -------------------------
# Build EVENT-month monthly targets
# -------------------------
monthly = build_monthly_event(klaim_raw)
truth_map = build_truth_map(monthly)

# ids Kaggle (15 baris target)
KAGGLE_IDS = sample_sub["id"].astype(str).tolist()

print("Monthly range:", monthly["year_month"].min(), "to", monthly["year_month"].max(), "| n_months =", len(monthly))
print("Sample submission ids (first 5):", KAGGLE_IDS[:5])

Monthly range: 2024-01 to 2025-07 | n_months = 19
Sample submission ids (first 5): ['2025_08_Claim_Frequency', '2025_08_Claim_Severity', '2025_08_Total_Claim', '2025_09_Claim_Frequency', '2025_09_Claim_Severity']


# Timeline Alignment (Event Month Only)

In [3]:
# =========================
# STAGE 2 — Timeline Alignment (Event Month Only)
# - Pakai "Tanggal Pasien Masuk RS" sebagai EVENT month
# - Validasi range bulan historis & siapkan helper untuk backtest splits
# =========================

import numpy as np
import pandas as pd

BASE_PATH = "/kaggle/input/datasets/dimaspashaakrilian/dsc-itb/"
PATH_KLAIM = BASE_PATH + "Data_Klaim.csv"
PATH_POLIS = BASE_PATH + "Data_Polis.csv"
PATH_SUB   = BASE_PATH + "sample_submission.csv"

# -------------------------
# Utils (ringan, aman)
# -------------------------
def clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
          .str.strip()
          .str.lower()
          .str.replace(r"\s+", "_", regex=True)
          .str.replace(r"[^a-z0-9_]+", "", regex=True)
    )
    return df

def parse_any_date_series(s: pd.Series) -> pd.Series:
    """
    Robust date parsing:
    - kalau sudah string ISO -> ok
    - kalau angka YYYYMMDD -> convert
    - fallback: to_datetime errors='coerce'
    """
    if pd.api.types.is_numeric_dtype(s):
        # asumsikan YYYYMMDD
        ss = s.astype("Int64").astype(str)
        # handle <NA>
        ss = ss.replace("<NA>", np.nan)
        return pd.to_datetime(ss, format="%Y%m%d", errors="coerce")
    else:
        return pd.to_datetime(s, errors="coerce")

def month_period(s: pd.Series) -> pd.Series:
    return s.dt.to_period("M")

def check_month_continuity(periods: pd.Series):
    """
    Validasi: apakah bulan berurutan tanpa loncat (boleh ada bulan kosong di data klaim? biasanya ada klaim setiap bulan).
    Ini tidak wajib fail, tapi kasih warning kalau ada gap.
    """
    ps = periods.dropna().sort_values().unique()
    if len(ps) == 0:
        raise ValueError("Tidak ada bulan valid setelah parsing tanggal.")
    # cek gap
    gaps = []
    for i in range(1, len(ps)):
        if ps[i] != ps[i-1] + 1:
            gaps.append((str(ps[i-1]), str(ps[i])))
    return ps, gaps

def period_to_str(p: pd.Period) -> str:
    return p.strftime("%Y-%m")

def build_event_month_column(klaim_df: pd.DataFrame) -> pd.DataFrame:
    df = clean_columns(klaim_df)

    col_event = "tanggal_pasien_masuk_rs"
    if col_event not in df.columns:
        raise ValueError("Kolom 'Tanggal Pasien Masuk RS' tidak ditemukan setelah clean. "
                         "Pastikan nama kolom di CSV sesuai.")

    df[col_event] = parse_any_date_series(df[col_event])

    # drop event date kosong (tidak bisa dipakai)
    n_before = len(df)
    df = df.dropna(subset=[col_event]).copy()
    n_after = len(df)

    # EVENT month sebagai Period
    df["event_month"] = month_period(df[col_event])
    df["event_ym"] = df["event_month"].astype(str)  # "YYYY-MM"

    print(f"[STAGE2] Parsed EVENT date: kept {n_after}/{n_before} rows (dropped {n_before-n_after} missing dates).")
    print(f"[STAGE2] EVENT month range: {df['event_ym'].min()} to {df['event_ym'].max()}")

    # continuity check
    ps, gaps = check_month_continuity(df["event_month"])
    if gaps:
        print("[STAGE2] WARNING: ada gap bulan (bulan tanpa klaim) atau parsing aneh:")
        for a,b in gaps[:10]:
            print("  gap:", a, "->", b)
    else:
        print("[STAGE2] EVENT months appear continuous (no gaps).")

    return df

def make_rolling_splits(event_months_sorted: list, horizon=5):
    """
    Buat rolling splits seperti:
      split0: train sampai month index T0, valid 5 bulan setelahnya
    event_months_sorted: list of "YYYY-MM" sudah urut naik
    """
    months = [pd.Period(m, freq="M") for m in event_months_sorted]
    months = sorted(set(months))
    # kita butuh minimal train_end + horizon <= last_month
    splits = []
    for train_end_idx in range(len(months)):
        train_end = months[train_end_idx]
        valid_start = train_end + 1
        valid_end = train_end + horizon
        if valid_end <= months[-1]:
            train_range = (months[0], train_end)
            valid_range = (valid_start, valid_end)
            splits.append((train_range, valid_range))
    return splits

# -------------------------
# Load
# -------------------------
klaim_raw = pd.read_csv(PATH_KLAIM)
polis_raw = pd.read_csv(PATH_POLIS)
sample_sub = pd.read_csv(PATH_SUB)

# -------------------------
# Apply Timeline Alignment
# -------------------------
klaim = build_event_month_column(klaim_raw)

# daftar bulan historis yang ada di klaim (event month)
hist_months = (
    klaim[["event_month"]]
    .dropna()
    .drop_duplicates()
    .sort_values("event_month")["event_month"]
    .astype(str)
    .tolist()
)

print(f"[STAGE2] Unique EVENT months: {len(hist_months)} months")
print("[STAGE2] Months:", hist_months[0], "->", hist_months[-1])

# -------------------------
# Prepare rolling splits for 5-step horizon backtest
# -------------------------
splits = make_rolling_splits(hist_months, horizon=5)

# Contoh: ambil 3 split terakhir (yang paling mirip test)
last3 = splits[-3:] if len(splits) >= 3 else splits
print("[STAGE2] Example rolling splits (train -> valid, horizon=5):")
for i, (tr, va) in enumerate(last3):
    print(f"  split{i}: Train={period_to_str(tr[0])}..{period_to_str(tr[1])} | "
          f"Valid={period_to_str(va[0])}..{period_to_str(va[1])}")

# -------------------------
# OPTIONAL: cek apakah "bulan target Kaggle" (2025-08..2025-12) memang AFTER hist end
# -------------------------
kaggle_target_months = ["2025-08","2025-09","2025-10","2025-11","2025-12"]
print("[STAGE2] Kaggle target months:", kaggle_target_months)
print("[STAGE2] Hist last month:", hist_months[-1])

[STAGE2] Parsed EVENT date: kept 4627/4627 rows (dropped 0 missing dates).
[STAGE2] EVENT month range: 2024-01 to 2025-07
[STAGE2] EVENT months appear continuous (no gaps).
[STAGE2] Unique EVENT months: 19 months
[STAGE2] Months: 2024-01 -> 2025-07
[STAGE2] Example rolling splits (train -> valid, horizon=5):
  split0: Train=2024-01..2024-12 | Valid=2025-01..2025-05
  split1: Train=2024-01..2025-01 | Valid=2025-02..2025-06
  split2: Train=2024-01..2025-02 | Valid=2025-03..2025-07
[STAGE2] Kaggle target months: ['2025-08', '2025-09', '2025-10', '2025-11', '2025-12']
[STAGE2] Hist last month: 2025-07


# Monthly Target Builder (Freq, Sev, Total)

In [4]:
# =========================
# STAGE 3 — Monthly Target Builder (Freq, Sev, Total)
# - Build target bulanan berbasis EVENT month (Tanggal Pasien Masuk RS)
# - Output: monthly (year_month, frequency, severity, total_claim)
# - Safety: handle freq=0, nilai negatif, dan sorting bulan
# =========================

import numpy as np
import pandas as pd

BASE_PATH = "/kaggle/input/datasets/dimaspashaakrilian/dsc-itb/"
PATH_KLAIM = BASE_PATH + "Data_Klaim.csv"
PATH_POLIS = BASE_PATH + "Data_Polis.csv"
PATH_SUB   = BASE_PATH + "sample_submission.csv"

# -------------------------
# Utils
# -------------------------
def clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
          .str.strip()
          .str.lower()
          .str.replace(r"\s+", "_", regex=True)
          .str.replace(r"[^a-z0-9_]+", "", regex=True)
    )
    return df

def parse_any_date_series(s: pd.Series) -> pd.Series:
    if pd.api.types.is_numeric_dtype(s):
        ss = s.astype("Int64").astype(str)
        ss = ss.replace("<NA>", np.nan)
        return pd.to_datetime(ss, format="%Y%m%d", errors="coerce")
    return pd.to_datetime(s, errors="coerce")

def safe_nonneg(x: pd.Series) -> pd.Series:
    # set nilai negatif jadi 0 (defensive)
    x = pd.to_numeric(x, errors="coerce")
    x = x.fillna(0.0)
    x = np.where(x < 0, 0.0, x)
    return pd.Series(x)

def build_monthly_targets_event(klaim_df: pd.DataFrame) -> pd.DataFrame:
    df = clean_columns(klaim_df)

    # kolom wajib (setelah clean)
    need = ["claim_id", "tanggal_pasien_masuk_rs", "nominal_klaim_yang_disetujui"]
    miss = [c for c in need if c not in df.columns]
    if miss:
        raise ValueError(f"Kolom wajib tidak ditemukan: {miss}. "
                         "Cek header CSV kamu dan pastikan sama seperti deskripsi.")

    # parse event date
    df["tanggal_pasien_masuk_rs"] = parse_any_date_series(df["tanggal_pasien_masuk_rs"])
    df = df.dropna(subset=["tanggal_pasien_masuk_rs"]).copy()

    # event month
    df["event_month"] = df["tanggal_pasien_masuk_rs"].dt.to_period("M")
    df["year_month"]  = df["event_month"].astype(str)  # "YYYY-MM"

    # nominal approved: defensive non-negative
    df["nominal_klaim_yang_disetujui"] = safe_nonneg(df["nominal_klaim_yang_disetujui"])

    # build monthly
    monthly = (
        df.groupby("year_month", as_index=False)
          .agg(
              frequency=("claim_id", "count"),
              total_claim=("nominal_klaim_yang_disetujui", "sum"),
          )
    )

    # sort by time (pakai Period agar aman)
    monthly["event_month"] = pd.PeriodIndex(monthly["year_month"], freq="M")
    monthly = monthly.sort_values("event_month").reset_index(drop=True)

    # severity
    monthly["severity"] = monthly["total_claim"] / monthly["frequency"].replace(0, np.nan)
    monthly["severity"] = monthly["severity"].fillna(0.0)

    # final columns order
    monthly = monthly[["year_month", "frequency", "severity", "total_claim"]].copy()

    return monthly

def validate_monthly(monthly: pd.DataFrame, expected_start=None, expected_end=None):
    # cek basic
    assert monthly["year_month"].is_monotonic_increasing, "year_month tidak urut naik."
    assert (monthly["frequency"] >= 0).all(), "Ada frequency negatif."
    assert (monthly["total_claim"] >= 0).all(), "Ada total_claim negatif."
    assert (monthly["severity"] >= 0).all(), "Ada severity negatif."

    # cek range opsional
    if expected_start is not None:
        if monthly["year_month"].min() != expected_start:
            print(f"[STAGE3] WARNING: start month {monthly['year_month'].min()} != expected {expected_start}")
    if expected_end is not None:
        if monthly["year_month"].max() != expected_end:
            print(f"[STAGE3] WARNING: end month {monthly['year_month'].max()} != expected {expected_end}")

    # cek konsistensi total ~ freq*sev (toleransi float)
    recon = monthly["frequency"] * monthly["severity"]
    rel_err = np.where(monthly["total_claim"] > 0,
                       np.abs(monthly["total_claim"] - recon) / monthly["total_claim"],
                       0.0)
    print("[STAGE3] Recon check: median rel_err =", float(np.median(rel_err)),
          "| max rel_err =", float(np.max(rel_err)))

# -------------------------
# Load
# -------------------------
klaim_raw = pd.read_csv(PATH_KLAIM)

# -------------------------
# Build Monthly Targets
# -------------------------
monthly = build_monthly_targets_event(klaim_raw)

print("[STAGE3] monthly shape:", monthly.shape)
print("[STAGE3] month range:", monthly["year_month"].min(), "to", monthly["year_month"].max())
display(monthly.head(3))
display(monthly.tail(3))

# -------------------------
# Validate
# -------------------------
validate_monthly(monthly, expected_start="2024-01", expected_end="2025-07")

[STAGE3] monthly shape: (19, 4)
[STAGE3] month range: 2024-01 to 2025-07


,year_month,frequency,severity,total_claim
0,2024-01,302,6.708934e+07,2.026098e+10
1,2024-02,208,6.663291e+07,1.385965e+10
2,2024-03,278,5.147935e+07,1.431126e+10


,year_month,frequency,severity,total_claim
16,2025-05,239,5.115814e+07,1.222680e+10
17,2025-06,234,5.715008e+07,1.337312e+10
18,2025-07,264,5.189101e+07,1.369923e+10


[STAGE3] Recon check: median rel_err = 0.0 | max rel_err = 0.0


# Data Integrity & Robustness

In [5]:
# =========================
# STAGE 4 — Data Integrity & Robustness
# - Clean + validate raw klaim & polis
# - Defensive fixes (dates, numerics, dedup)
# - Build robust monthly targets (EVENT month)
# - Freeze "safe statistics" (corridors) untuk guardrails tahap berikutnya
# =========================

import numpy as np
import pandas as pd
from IPython.display import display

BASE_PATH = "/kaggle/input/datasets/dimaspashaakrilian/dsc-itb/"
PATH_KLAIM = BASE_PATH + "Data_Klaim.csv"
PATH_POLIS = BASE_PATH + "Data_Polis.csv"
PATH_SUB   = BASE_PATH + "sample_submission.csv"

# -------------------------
# Utils
# -------------------------
def clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
          .str.strip()
          .str.lower()
          .str.replace(r"\s+", "_", regex=True)
          .str.replace(r"[^a-z0-9_]+", "", regex=True)
    )
    return df

def parse_any_date_series(s: pd.Series) -> pd.Series:
    # robust: numeric YYYYMMDD or string date
    if pd.api.types.is_numeric_dtype(s):
        ss = s.astype("Int64").astype(str).replace("<NA>", np.nan)
        return pd.to_datetime(ss, format="%Y%m%d", errors="coerce")
    return pd.to_datetime(s, errors="coerce")

def to_num_nonneg(s: pd.Series) -> pd.Series:
    x = pd.to_numeric(s, errors="coerce").fillna(0.0).astype(float)
    x = np.where(np.isfinite(x), x, 0.0)
    x = np.where(x < 0, 0.0, x)
    return pd.Series(x, index=s.index)

def safe_quantiles(x: pd.Series, q_low=0.01, q_high=0.99):
    x = pd.to_numeric(x, errors="coerce").astype(float)
    x = x.replace([np.inf, -np.inf], np.nan).dropna()
    if len(x) == 0:
        return (0.0, 0.0)
    return (float(x.quantile(q_low)), float(x.quantile(q_high)))

def build_monthly_targets_event(klaim_df: pd.DataFrame) -> pd.DataFrame:
    df = clean_columns(klaim_df)

    need = ["claim_id", "tanggal_pasien_masuk_rs", "nominal_klaim_yang_disetujui"]
    miss = [c for c in need if c not in df.columns]
    if miss:
        raise ValueError(f"Kolom wajib tidak ditemukan: {miss}")

    df["tanggal_pasien_masuk_rs"] = parse_any_date_series(df["tanggal_pasien_masuk_rs"])
    df = df.dropna(subset=["tanggal_pasien_masuk_rs"]).copy()

    df["nominal_klaim_yang_disetujui"] = to_num_nonneg(df["nominal_klaim_yang_disetujui"])

    df["event_month"] = df["tanggal_pasien_masuk_rs"].dt.to_period("M")
    df["year_month"] = df["event_month"].astype(str)

    monthly = (
        df.groupby("year_month", as_index=False)
          .agg(
              frequency=("claim_id", "count"),
              total_claim=("nominal_klaim_yang_disetujui", "sum"),
          )
    )
    monthly["event_month"] = pd.PeriodIndex(monthly["year_month"], freq="M")
    monthly = monthly.sort_values("event_month").reset_index(drop=True)

    monthly["severity"] = monthly["total_claim"] / monthly["frequency"].replace(0, np.nan)
    monthly["severity"] = monthly["severity"].fillna(0.0)

    return monthly[["year_month", "frequency", "severity", "total_claim"]].copy()

def validate_monthly(monthly: pd.DataFrame):
    # monotonic check
    pm = pd.PeriodIndex(monthly["year_month"], freq="M")
    if not all(pm[i] <= pm[i+1] for i in range(len(pm)-1)):
        raise ValueError("Monthly year_month tidak urut naik.")

    # non-neg
    for c in ["frequency", "severity", "total_claim"]:
        if (monthly[c] < 0).any():
            raise ValueError(f"Ada nilai negatif pada {c}.")

    # recon check
    recon = monthly["frequency"] * monthly["severity"]
    rel_err = np.where(monthly["total_claim"] > 0,
                       np.abs(monthly["total_claim"] - recon) / monthly["total_claim"],
                       0.0)
    print("[STAGE4] Recon check (Total vs Freq*Sev): "
          f"median={float(np.median(rel_err)):.6f} | max={float(np.max(rel_err)):.6f}")

def compute_active_policies_by_month(polis_df: pd.DataFrame, months_ym: list) -> pd.Series:
    """
    True exposure sederhana: jumlah polis dengan tanggal_efektif_polis <= akhir bulan.
    Catatan: kalau semua polis sudah efektif jauh sebelum 2024, exposure akan konstan.
    """
    p = clean_columns(polis_df)
    if "tanggal_efektif_polis" not in p.columns:
        return pd.Series([np.nan]*len(months_ym), index=months_ym, name="active_policies")

    eff = parse_any_date_series(p["tanggal_efektif_polis"])
    eff = eff.dropna()

    out = []
    for ym in months_ym:
        per = pd.Period(ym, freq="M")
        month_end = per.end_time  # timestamp end-of-month
        out.append(int((eff <= month_end).sum()))
    return pd.Series(out, index=months_ym, name="active_policies")

# -------------------------
# Load raw
# -------------------------
klaim_raw = pd.read_csv(PATH_KLAIM)
polis_raw = pd.read_csv(PATH_POLIS)
sample_sub = pd.read_csv(PATH_SUB)

# -------------------------
# Clean & robustify klaim
# -------------------------
klaim = clean_columns(klaim_raw)

# minimal required columns check
req_cols = [
    "claim_id", "nomor_polis",
    "tanggal_pasien_masuk_rs",
    "nominal_klaim_yang_disetujui"
]
missing = [c for c in req_cols if c not in klaim.columns]
if missing:
    raise ValueError(f"[STAGE4] Kolom wajib hilang di Data_Klaim.csv: {missing}")

# Dedup claim_id (defensive)
n0 = len(klaim)
klaim = klaim.drop_duplicates(subset=["claim_id"], keep="first").copy()
n1 = len(klaim)
print(f"[STAGE4] Dedup Claim ID: {n1}/{n0} rows kept (dropped {n0-n1}).")

# Parse tanggal penting
klaim["tanggal_pasien_masuk_rs"]  = parse_any_date_series(klaim["tanggal_pasien_masuk_rs"])
if "tanggal_pasien_keluar_rs" in klaim.columns:
    klaim["tanggal_pasien_keluar_rs"] = parse_any_date_series(klaim["tanggal_pasien_keluar_rs"])
if "tanggal_pembayaran_klaim" in klaim.columns:
    klaim["tanggal_pembayaran_klaim"] = parse_any_date_series(klaim["tanggal_pembayaran_klaim"])

# Numeric robust
klaim["nominal_klaim_yang_disetujui"] = to_num_nonneg(klaim["nominal_klaim_yang_disetujui"])
if "nominal_biaya_rs_yang_terjadi" in klaim.columns:
    klaim["nominal_biaya_rs_yang_terjadi"] = to_num_nonneg(klaim["nominal_biaya_rs_yang_terjadi"])

# Derived: length of stay (optional; useful untuk EDA, bukan untuk timeline)
if "tanggal_pasien_keluar_rs" in klaim.columns:
    los = (klaim["tanggal_pasien_keluar_rs"] - klaim["tanggal_pasien_masuk_rs"]).dt.days
    los = los.replace([np.inf, -np.inf], np.nan)
    # defensive clip: LOS negatif -> 0, terlalu besar -> cap 365
    los = los.fillna(0).astype(int)
    los = np.clip(los, 0, 365)
    klaim["los_days"] = los

# Derived: approval ratio (optional)
if "nominal_biaya_rs_yang_terjadi" in klaim.columns:
    bill = klaim["nominal_biaya_rs_yang_terjadi"].replace(0, np.nan)
    ratio = klaim["nominal_klaim_yang_disetujui"] / bill
    ratio = ratio.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    # defensive clip ratio ke [0, 3] (outlier extreme biasanya noise input)
    klaim["approved_to_bill_ratio"] = np.clip(ratio, 0.0, 3.0)

# Drop rows tanpa EVENT date (wajib untuk timeline)
before = len(klaim)
klaim = klaim.dropna(subset=["tanggal_pasien_masuk_rs"]).copy()
after = len(klaim)
print(f"[STAGE4] EVENT date valid: {after}/{before} rows kept (dropped {before-after}).")

# Quick integrity summary
print("[STAGE4] EVENT date range:",
      klaim["tanggal_pasien_masuk_rs"].min(), "to", klaim["tanggal_pasien_masuk_rs"].max())
print("[STAGE4] Approved amount: min/median/max =",
      float(klaim["nominal_klaim_yang_disetujui"].min()),
      float(klaim["nominal_klaim_yang_disetujui"].median()),
      float(klaim["nominal_klaim_yang_disetujui"].max()))

# -------------------------
# Build robust monthly targets (EVENT)
# -------------------------
monthly = build_monthly_targets_event(klaim)
print("[STAGE4] Monthly shape:", monthly.shape)
print("[STAGE4] Monthly range:", monthly["year_month"].min(), "to", monthly["year_month"].max())
display(monthly.head(3))
display(monthly.tail(3))
validate_monthly(monthly)

# -------------------------
# Exposure (active policies) by month (from polis)
# -------------------------
months_ym = monthly["year_month"].tolist()
active_policies = compute_active_policies_by_month(polis_raw, months_ym)
print("[STAGE4] Active policies (min/max):", int(np.nanmin(active_policies)), int(np.nanmax(active_policies)))
display(pd.DataFrame({"year_month": months_ym, "active_policies": active_policies.values}).head(3))

# -------------------------
# Freeze SAFE statistics for later guardrails
# (gunakan TRAIN-only pada backtest nanti; ini versi global dari seluruh historis)
# -------------------------
STAGE4_STATS = {}

# per-claim corridor (approved)
STAGE4_STATS["claim_approved_q01_q99"] = safe_quantiles(klaim["nominal_klaim_yang_disetujui"], 0.01, 0.99)

# monthly corridors
STAGE4_STATS["monthly_frequency_q05_q95"] = safe_quantiles(monthly["frequency"], 0.05, 0.95)
STAGE4_STATS["monthly_total_q05_q95"]     = safe_quantiles(monthly["total_claim"], 0.05, 0.95)
STAGE4_STATS["monthly_severity_q05_q95"]  = safe_quantiles(monthly["severity"], 0.05, 0.95)

# log corridors (lebih stabil untuk guardrail)
STAGE4_STATS["monthly_log_total_q05_q95"]    = safe_quantiles(np.log1p(monthly["total_claim"]), 0.05, 0.95)
STAGE4_STATS["monthly_log_severity_q05_q95"] = safe_quantiles(np.log1p(monthly["severity"]), 0.05, 0.95)

# exposure info
STAGE4_STATS["active_policies_min_max"] = (int(np.nanmin(active_policies)), int(np.nanmax(active_policies)))

print("[STAGE4] STAGE4_STATS frozen:")
for k, v in STAGE4_STATS.items():
    print("  -", k, "=", v)

# monthly + klaim bersih + STAGE4_STATS siap dipakai tahap berikutnya

[STAGE4] Dedup Claim ID: 4627/4627 rows kept (dropped 0).
[STAGE4] EVENT date valid: 4627/4627 rows kept (dropped 0).
[STAGE4] EVENT date range: 2024-01-01 00:00:00 to 2025-07-31 00:00:00
[STAGE4] Approved amount: min/median/max = 0.0 14467899.0 2197500000.0
[STAGE4] Monthly shape: (19, 4)
[STAGE4] Monthly range: 2024-01 to 2025-07


,year_month,frequency,severity,total_claim
0,2024-01,302,6.708934e+07,2.026098e+10
1,2024-02,208,6.663291e+07,1.385965e+10
2,2024-03,278,5.147935e+07,1.431126e+10


,year_month,frequency,severity,total_claim
16,2025-05,239,5.115814e+07,1.222680e+10
17,2025-06,234,5.715008e+07,1.337312e+10
18,2025-07,264,5.189101e+07,1.369923e+10


[STAGE4] Recon check (Total vs Freq*Sev): median=0.000000 | max=0.000000
[STAGE4] Active policies (min/max): 4096 4096


,year_month,active_policies
0,2024-01,4096
1,2024-02,4096
2,2024-03,4096


[STAGE4] STAGE4_STATS frozen:
  - claim_approved_q01_q99 = (187241.8, 633686130.0819994)
  - monthly_frequency_q05_q95 = (208.0, 280.4)
  - monthly_total_q05_q95 = (11008861478.031, 17758584483.70231)
  - monthly_severity_q05_q95 = (46102714.36698347, 67486319.13475524)
  - monthly_log_total_q05_q95 = (23.120995114628702, 23.59911498636526)
  - monthly_log_severity_q05_q95 = (17.646313196094273, 18.02728445902302)
  - active_policies_min_max = (4096, 4096)


# Portfolio Decomposition (Mixture-Aware)

In [6]:
# =========================
# STAGE 5 — Portfolio Decomposition (Mixture-Aware) [REVISED]
# Fix: build_full_month_index pakai pd.period_range (menghindari error MonthEnd + int)
# =========================

import numpy as np
import pandas as pd
from IPython.display import display

BASE_PATH = "/kaggle/input/datasets/dimaspashaakrilian/dsc-itb/"
PATH_KLAIM = BASE_PATH + "Data_Klaim.csv"

# -------------------------
# Utils
# -------------------------
def clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
          .str.strip()
          .str.lower()
          .str.replace(r"\s+", "_", regex=True)
          .str.replace(r"[^a-z0-9_]+", "", regex=True)
    )
    return df

def parse_any_date_series(s: pd.Series) -> pd.Series:
    if pd.api.types.is_numeric_dtype(s):
        ss = s.astype("Int64").astype(str).replace("<NA>", np.nan)
        return pd.to_datetime(ss, format="%Y%m%d", errors="coerce")
    return pd.to_datetime(s, errors="coerce")

def to_num_nonneg(s: pd.Series) -> pd.Series:
    x = pd.to_numeric(s, errors="coerce").fillna(0.0).astype(float)
    x = np.where(np.isfinite(x), x, 0.0)
    x = np.where(x < 0, 0.0, x)
    return pd.Series(x, index=s.index)

def build_full_month_index(min_ym: str, max_ym: str) -> list:
    """
    min_ym, max_ym: "YYYY-MM"
    return list of "YYYY-MM" lengkap dan berurutan.
    """
    pr = pd.period_range(start=min_ym, end=max_ym, freq="M")
    return [p.strftime("%Y-%m") for p in pr]

# -------------------------
# Ambil klaim bersih dari STAGE 4 kalau ada, kalau tidak -> build minimal di sini
# -------------------------
if "klaim" in globals():
    df = klaim.copy()
    df = clean_columns(df)
else:
    df = pd.read_csv(PATH_KLAIM)
    df = clean_columns(df)

# kolom wajib
need = ["claim_id", "tanggal_pasien_masuk_rs", "nominal_klaim_yang_disetujui"]
miss = [c for c in need if c not in df.columns]
if miss:
    raise ValueError(f"[STAGE5] Kolom wajib tidak ada: {miss}")

# EVENT month
df["tanggal_pasien_masuk_rs"] = parse_any_date_series(df["tanggal_pasien_masuk_rs"])
df = df.dropna(subset=["tanggal_pasien_masuk_rs"]).copy()
df["year_month"] = df["tanggal_pasien_masuk_rs"].dt.to_period("M").astype(str)

# nominal approved
df["nominal_klaim_yang_disetujui"] = to_num_nonneg(df["nominal_klaim_yang_disetujui"])

# -------------------------
# Definisi grup SG vs NONSG
# -------------------------
if "lokasi_rs" not in df.columns:
    raise ValueError("[STAGE5] Kolom 'Lokasi RS' tidak ditemukan (setelah clean menjadi 'lokasi_rs').")

loc = df["lokasi_rs"].astype(str).str.strip().str.lower()
is_sg = loc.eq("singapore") | loc.str.contains(r"\bsingapore\b", regex=True)

df["grp"] = np.where(is_sg, "SG", "NONSG")

print("[STAGE5] Group share (rows):")
display(df["grp"].value_counts(dropna=False).to_frame("n_rows"))

# -------------------------
# Build monthly per group (full grid months x groups)
# -------------------------
g = (
    df.groupby(["year_month", "grp"], as_index=False)
      .agg(
          frequency=("claim_id", "count"),
          total_claim=("nominal_klaim_yang_disetujui", "sum"),
      )
)

# full grid (bulan lengkap x grup)
min_ym = g["year_month"].min()
max_ym = g["year_month"].max()
all_months = build_full_month_index(min_ym, max_ym)
all_groups = ["SG", "NONSG"]

full_idx = pd.MultiIndex.from_product([all_months, all_groups], names=["year_month", "grp"])
g = g.set_index(["year_month", "grp"]).reindex(full_idx).reset_index()

g["frequency"] = g["frequency"].fillna(0).astype(float)
g["total_claim"] = g["total_claim"].fillna(0.0).astype(float)
g["severity"] = np.where(g["frequency"] > 0, g["total_claim"] / g["frequency"], 0.0)

print("[STAGE5] Panel long (per month x group) sample:")
display(g.head(6))

# -------------------------
# Pivot ke wide
# -------------------------
panel_wide = (
    g.pivot(index="year_month", columns="grp", values=["frequency", "total_claim", "severity"])
     .sort_index()
)
panel_wide.columns = [f"{a.lower()}_{b.lower()}" for a, b in panel_wide.columns]
panel_wide = panel_wide.reset_index()

# portfolio totals dari penjumlahan grup
panel_wide["frequency_portfolio"] = panel_wide["frequency_sg"] + panel_wide["frequency_nonsg"]
panel_wide["total_claim_portfolio"] = panel_wide["total_claim_sg"] + panel_wide["total_claim_nonsg"]
panel_wide["severity_portfolio"] = np.where(
    panel_wide["frequency_portfolio"] > 0,
    panel_wide["total_claim_portfolio"] / panel_wide["frequency_portfolio"],
    0.0
)

print("[STAGE5] Panel wide sample:")
display(panel_wide.head(3))
display(panel_wide.tail(3))

# -------------------------
# Validasi vs 'monthly' dari STAGE 3/4 (kalau ada)
# -------------------------
if "monthly" in globals():
    m = monthly.copy()
    m = clean_columns(m)
    need_m = ["year_month", "frequency", "total_claim"]
    miss_m = [c for c in need_m if c not in m.columns]
    if miss_m:
        print("[STAGE5] WARNING: monthly ada tapi kolomnya tidak sesuai:", miss_m)
    else:
        chk = panel_wide.merge(m[["year_month","frequency","total_claim"]], on="year_month", how="left", suffixes=("", "_orig"))
        freq_rel = np.where(chk["frequency"] > 0,
                            np.abs(chk["frequency_portfolio"] - chk["frequency"]) / chk["frequency"],
                            0.0)
        tot_rel  = np.where(chk["total_claim"] > 0,
                            np.abs(chk["total_claim_portfolio"] - chk["total_claim"]) / chk["total_claim"],
                            0.0)
        print("[STAGE5] Validate vs monthly (portfolio):")
        print("  Freq  rel_err median/max =", float(np.median(freq_rel)), float(np.max(freq_rel)))
        print("  Total rel_err median/max =", float(np.median(tot_rel)),  float(np.max(tot_rel)))
else:
    print("[STAGE5] INFO: variabel 'monthly' belum ada. Validasi ke monthly dilewati.")

# -------------------------
# Simpan artifact
# -------------------------
STAGE5_PANEL_LONG = g.copy()
STAGE5_PANEL_WIDE = panel_wide.copy()
print("[STAGE5] Stored: STAGE5_PANEL_LONG, STAGE5_PANEL_WIDE")

[STAGE5] Group share (rows):


,n_rows
grp,
NONSG,3608
SG,1019


[STAGE5] Panel long (per month x group) sample:


,year_month,grp,frequency,total_claim,severity
0,2024-01,SG,62.0,1.031967e+10,1.664463e+08
1,2024-01,NONSG,240.0,9.941314e+09,4.142214e+07
2,2024-02,SG,51.0,8.125130e+09,1.593163e+08
3,2024-02,NONSG,157.0,5.734516e+09,3.652558e+07
4,2024-03,SG,61.0,5.800471e+09,9.508969e+07
5,2024-03,NONSG,217.0,8.510787e+09,3.922022e+07


[STAGE5] Panel wide sample:


,year_month,frequency_nonsg,frequency_sg,total_claim_nonsg,total_claim_sg,severity_nonsg,severity_sg,frequency_portfolio,total_claim_portfolio,severity_portfolio
0,2024-01,240.0,62.0,9.941314e+09,1.031967e+10,4.142214e+07,1.664463e+08,302.0,2.026098e+10,6.708934e+07
1,2024-02,157.0,51.0,5.734516e+09,8.125130e+09,3.652558e+07,1.593163e+08,208.0,1.385965e+10,6.663291e+07
2,2024-03,217.0,61.0,8.510787e+09,5.800471e+09,3.922022e+07,9.508969e+07,278.0,1.431126e+10,5.147935e+07


,year_month,frequency_nonsg,frequency_sg,total_claim_nonsg,total_claim_sg,severity_nonsg,severity_sg,frequency_portfolio,total_claim_portfolio,severity_portfolio
16,2025-05,186.0,53.0,7.772707e+09,4.454088e+09,4.178875e+07,8.403940e+07,239.0,1.222680e+10,5.115814e+07
17,2025-06,175.0,59.0,5.894068e+09,7.479052e+09,3.368039e+07,1.267636e+08,234.0,1.337312e+10,5.715008e+07
18,2025-07,221.0,43.0,8.639742e+09,5.059485e+09,3.909385e+07,1.176624e+08,264.0,1.369923e+10,5.189101e+07


[STAGE5] Validate vs monthly (portfolio):
  Freq  rel_err median/max = 0.0 0.0
  Total rel_err median/max = 0.0 1.5730493480979623e-16
[STAGE5] Stored: STAGE5_PANEL_LONG, STAGE5_PANEL_WIDE


# Model Core (Multiplicative Forecasting)

In [7]:
# =========================
# STAGE 6 — Model Core (Multiplicative Forecasting)
# - Kandidat model ringan untuk data bulanan pendek:
#   (1) YoY Anchor (seasonal-naive + growth)
#   (2) Holt/ETS Trend (tanpa seasonal, karena titik data < 24 bulan)
#   (3) Recent Mean / EWM (baseline konservatif)
# - Forecast dilakukan untuk seri per-grup: SG vs NONSG
# - Total diprediksi di log1p-space (multiplicative-friendly untuk MAPE)
# =========================

import numpy as np
import pandas as pd

# statsmodels untuk Holt/ETS
try:
    from statsmodels.tsa.holtwinters import ExponentialSmoothing
except Exception:
    # di Kaggle biasanya sudah ada; kalau belum, uncomment:
    # !pip -q install statsmodels
    from statsmodels.tsa.holtwinters import ExponentialSmoothing

# -------------------------
# Helpers
# -------------------------
def ym_to_period(ym: str) -> pd.Period:
    return pd.Period(ym, freq="M")

def period_to_ym(p: pd.Period) -> str:
    return p.strftime("%Y-%m")

def make_future_months(last_ym: str, horizon=5) -> list:
    p0 = ym_to_period(last_ym) + 1
    pr = pd.period_range(start=p0, periods=horizon, freq="M")
    return [period_to_ym(p) for p in pr]

def safe_nonneg_array(x):
    x = np.asarray(x, dtype=float)
    x = np.where(np.isfinite(x), x, 0.0)
    x = np.where(x < 0, 0.0, x)
    return x

def series_from_panel(panel_wide: pd.DataFrame, col: str) -> pd.Series:
    s = panel_wide.set_index("year_month")[col].astype(float)
    s = s.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    s = pd.Series(safe_nonneg_array(s.values), index=s.index)
    return s

# -------------------------
# Model 1: YoY Anchor (seasonal naive + growth factor)
# -------------------------
def forecast_yoy_anchor(y: pd.Series, future_months: list, growth_clip=(0.7, 1.3)) -> pd.Series:
    """
    y: indexed by "YYYY-MM"
    forecast(m) = y(m-12) * g
    g dihitung dari median ratio y(t)/y(t-12) pada overlap historis.
    fallback jika m-12 tidak ada: pakai median y 3 bulan terakhir
    """
    idx = [ym_to_period(m) for m in y.index]
    y_period = pd.Series(y.values, index=idx).sort_index()

    # hitung growth dari overlap (t dan t-12)
    ratios = []
    for p in y_period.index:
        p_prev = p - 12
        if p_prev in y_period.index and y_period.loc[p_prev] > 0:
            ratios.append(y_period.loc[p] / y_period.loc[p_prev])
    if len(ratios) >= 3:
        g = float(np.median(ratios))
    else:
        g = 1.0
    g = float(np.clip(g, growth_clip[0], growth_clip[1]))

    # baseline fallback
    tail = y_period.iloc[-3:] if len(y_period) >= 3 else y_period
    fallback = float(np.median(tail.values)) if len(tail) else 0.0

    preds = []
    for m in future_months:
        p = ym_to_period(m)
        ref = p - 12
        if ref in y_period.index:
            base = float(y_period.loc[ref])
        else:
            base = fallback
        preds.append(max(0.0, base * g))

    return pd.Series(preds, index=future_months)

# -------------------------
# Model 2: Holt/ETS Trend (no seasonal)
# - Total: fit on log1p(Total) then expm1
# - Freq: fit on level (optionally log1p if kamu mau, tapi di sini level)
# -------------------------
def forecast_holt_trend(y: pd.Series, horizon=5, use_log=False) -> np.ndarray:
    """
    Robust Holt trend:
    - Kalau data terlalu pendek -> fallback ke mean terakhir
    """
    yy = y.astype(float).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    yy = pd.Series(safe_nonneg_array(yy.values), index=yy.index)

    if len(yy) < 4:
        base = float(np.mean(yy.values)) if len(yy) else 0.0
        return np.array([base]*horizon, dtype=float)

    if use_log:
        z = np.log1p(yy.values)
        fit = ExponentialSmoothing(
            z, trend="add", seasonal=None, damped_trend=True,
            initialization_method="estimated"
        ).fit(optimized=True)
        fc = fit.forecast(horizon)
        out = np.expm1(np.asarray(fc, dtype=float))
        return safe_nonneg_array(out)

    else:
        fit = ExponentialSmoothing(
            yy.values, trend="add", seasonal=None, damped_trend=True,
            initialization_method="estimated"
        ).fit(optimized=True)
        fc = fit.forecast(horizon)
        out = np.asarray(fc, dtype=float)
        return safe_nonneg_array(out)

# -------------------------
# Model 3: Conservative baseline (Recent Mean / EWM)
# -------------------------
def forecast_recent_mean(y: pd.Series, horizon=5, window=3) -> np.ndarray:
    yy = y.astype(float).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    yy = pd.Series(safe_nonneg_array(yy.values), index=yy.index)

    if len(yy) == 0:
        return np.zeros(horizon, dtype=float)

    w = min(window, len(yy))
    base = float(np.mean(yy.iloc[-w:].values))
    return np.array([max(0.0, base)]*horizon, dtype=float)

# -------------------------
# Input: STAGE5_PANEL_WIDE harus sudah ada
# -------------------------
if "STAGE5_PANEL_WIDE" not in globals():
    raise ValueError("STAGE5_PANEL_WIDE tidak ditemukan. Jalankan STAGE 5 dulu sampai stored.")

panel_wide = STAGE5_PANEL_WIDE.copy()

# pastikan kolom ada
need_cols = ["year_month", "frequency_sg", "total_claim_sg", "frequency_nonsg", "total_claim_nonsg"]
miss = [c for c in need_cols if c not in panel_wide.columns]
if miss:
    raise ValueError(f"Kolom STAGE5_PANEL_WIDE belum lengkap: {miss}")

panel_wide = panel_wide.sort_values("year_month").reset_index(drop=True)

last_ym = panel_wide["year_month"].iloc[-1]
future_months = make_future_months(last_ym, horizon=5)

print("[STAGE6] Hist last month:", last_ym)
print("[STAGE6] Future months:", future_months)

# seri per grup
SERIES = {
    "freq_sg":    series_from_panel(panel_wide, "frequency_sg"),
    "total_sg":   series_from_panel(panel_wide, "total_claim_sg"),
    "freq_nonsg": series_from_panel(panel_wide, "frequency_nonsg"),
    "total_nonsg":series_from_panel(panel_wide, "total_claim_nonsg"),
}

# -------------------------
# Build candidate forecasts
# -------------------------
STAGE6_CANDIDATES = {}  # dict[series_name][model_name] = pd.Series(future_months)

for name, s in SERIES.items():
    is_total = name.startswith("total")

    # YoY anchor (bagus untuk seasonal pattern + growth)
    pred_yoy = forecast_yoy_anchor(s, future_months, growth_clip=(0.6, 1.5))

    # Holt trend (untuk total pakai log1p)
    pred_holt = pd.Series(
        forecast_holt_trend(s, horizon=len(future_months), use_log=is_total),
        index=future_months
    )

    # Conservative baseline
    pred_mean = pd.Series(
        forecast_recent_mean(s, horizon=len(future_months), window=3),
        index=future_months
    )

    # simpan
    STAGE6_CANDIDATES[name] = {
        "yoy_anchor": pred_yoy,
        "holt_trend": pred_holt,
        "recent_mean": pred_mean,
    }

print("[STAGE6] Stored: STAGE6_CANDIDATES (per series, 3 model candidates)")

# preview ringkas (tabel)
def preview_candidates(series_key: str):
    d = STAGE6_CANDIDATES[series_key]
    out = pd.DataFrame({k: v.values for k, v in d.items()}, index=future_months)
    out.index.name = "year_month"
    return out

print("\n[STAGE6] Preview candidates: freq_sg")
display(preview_candidates("freq_sg"))

print("\n[STAGE6] Preview candidates: total_sg")
display(preview_candidates("total_sg"))

[STAGE6] Hist last month: 2025-07
[STAGE6] Future months: ['2025-08', '2025-09', '2025-10', '2025-11', '2025-12']
[STAGE6] Stored: STAGE6_CANDIDATES (per series, 3 model candidates)

[STAGE6] Preview candidates: freq_sg


,yoy_anchor,holt_trend,recent_mean
year_month,,,
2025-08,38.360656,52.485092,51.666667
2025-09,42.295082,52.471602,51.666667
2025-10,61.967213,52.460809,51.666667
2025-11,78.688525,52.452175,51.666667
2025-12,42.295082,52.445268,51.666667



[STAGE6] Preview candidates: total_sg


,yoy_anchor,holt_trend,recent_mean
year_month,,,
2025-08,6.832506e+09,5.982057e+09,5.664208e+09
2025-09,6.747276e+09,5.977077e+09,5.664208e+09
2025-10,6.729731e+09,5.973095e+09,5.664208e+09
2025-11,8.174522e+09,5.969912e+09,5.664208e+09
2025-12,8.186108e+09,5.967367e+09,5.664208e+09


# Rolling Backtest (5-Step Horizon)

In [8]:
# =========================
# STAGE 7 — Rolling Backtest (5-Step Horizon)
# - Backtest horizon 5 bulan (meniru target Kaggle)
# - Evaluasi kandidat model family: yoy_anchor, holt_trend, recent_mean
# - Forecast dilakukan per-grup (SG vs NONSG) lalu digabung ke portfolio
# - Skor = avg(MAPE_freq, MAPE_sev, MAPE_total)
# =========================

import numpy as np
import pandas as pd
from IPython.display import display

try:
    from statsmodels.tsa.holtwinters import ExponentialSmoothing
except Exception:
    # jika statsmodels belum ada di notebook kamu, uncomment:
    # !pip -q install statsmodels
    from statsmodels.tsa.holtwinters import ExponentialSmoothing

# -------------------------
# Safety / utils
# -------------------------
EPS = 1e-9

def mape(y_true, y_pred, eps=EPS) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.maximum(np.abs(y_true), eps)
    return float(np.mean(np.abs((y_true - y_pred) / denom)))

def ym_to_period(ym: str) -> pd.Period:
    return pd.Period(ym, freq="M")

def period_to_ym(p: pd.Period) -> str:
    return p.strftime("%Y-%m")

def make_rolling_splits(months_ym: list, horizon=5):
    months = [ym_to_period(m) for m in months_ym]
    months = sorted(months)
    splits = []
    for i in range(len(months)):
        train_end = months[i]
        valid_start = train_end + 1
        valid_end = train_end + horizon
        if valid_end <= months[-1]:
            tr = (months[0], train_end)
            va = (valid_start, valid_end)
            splits.append((tr, va))
    return splits

def safe_nonneg_array(x):
    x = np.asarray(x, dtype=float)
    x = np.where(np.isfinite(x), x, 0.0)
    x = np.where(x < 0, 0.0, x)
    return x

def forecast_yoy_anchor(y: pd.Series, future_months: list, growth_clip=(0.6, 1.5)) -> pd.Series:
    # y index: "YYYY-MM"
    idx = [ym_to_period(m) for m in y.index]
    yP = pd.Series(y.values, index=idx).sort_index()

    ratios = []
    for p in yP.index:
        p_prev = p - 12
        if p_prev in yP.index and yP.loc[p_prev] > 0:
            ratios.append(yP.loc[p] / yP.loc[p_prev])
    g = float(np.median(ratios)) if len(ratios) >= 3 else 1.0
    g = float(np.clip(g, growth_clip[0], growth_clip[1]))

    tail = yP.iloc[-3:] if len(yP) >= 3 else yP
    fallback = float(np.median(tail.values)) if len(tail) else 0.0

    preds = []
    for m in future_months:
        p = ym_to_period(m)
        ref = p - 12
        base = float(yP.loc[ref]) if ref in yP.index else fallback
        preds.append(max(0.0, base * g))
    return pd.Series(preds, index=future_months)

def forecast_holt_trend(y: pd.Series, horizon=5, use_log=False) -> np.ndarray:
    yy = y.astype(float).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    yy = pd.Series(safe_nonneg_array(yy.values), index=yy.index)

    if len(yy) < 4:
        base = float(np.mean(yy.values)) if len(yy) else 0.0
        return np.array([base]*horizon, dtype=float)

    if use_log:
        z = np.log1p(yy.values)
        fit = ExponentialSmoothing(
            z, trend="add", seasonal=None, damped_trend=True,
            initialization_method="estimated"
        ).fit(optimized=True)
        fc = np.asarray(fit.forecast(horizon), dtype=float)
        return safe_nonneg_array(np.expm1(fc))

    fit = ExponentialSmoothing(
        yy.values, trend="add", seasonal=None, damped_trend=True,
        initialization_method="estimated"
    ).fit(optimized=True)
    fc = np.asarray(fit.forecast(horizon), dtype=float)
    return safe_nonneg_array(fc)

def forecast_recent_mean(y: pd.Series, horizon=5, window=3) -> np.ndarray:
    yy = y.astype(float).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    yy = pd.Series(safe_nonneg_array(yy.values), index=yy.index)
    if len(yy) == 0:
        return np.zeros(horizon, dtype=float)
    w = min(window, len(yy))
    base = float(np.mean(yy.iloc[-w:].values))
    return np.array([max(0.0, base)]*horizon, dtype=float)

def get_series(panel: pd.DataFrame, col: str, train_months: list) -> pd.Series:
    s = panel.set_index("year_month").loc[train_months, col].astype(float)
    s = s.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    s = pd.Series(safe_nonneg_array(s.values), index=s.index)
    return s

def predict_family_for_split(panel: pd.DataFrame, train_months: list, valid_months: list, family: str):
    # ambil seri train per grup
    freq_sg_tr    = get_series(panel, "frequency_sg", train_months)
    total_sg_tr   = get_series(panel, "total_claim_sg", train_months)
    freq_ns_tr    = get_series(panel, "frequency_nonsg", train_months)
    total_ns_tr   = get_series(panel, "total_claim_nonsg", train_months)

    H = len(valid_months)

    # pred per grup
    if family == "yoy_anchor":
        pf_sg  = forecast_yoy_anchor(freq_sg_tr,  valid_months)
        pt_sg  = forecast_yoy_anchor(total_sg_tr, valid_months)
        pf_ns  = forecast_yoy_anchor(freq_ns_tr,  valid_months)
        pt_ns  = forecast_yoy_anchor(total_ns_tr, valid_months)

    elif family == "holt_trend":
        pf_sg  = pd.Series(forecast_holt_trend(freq_sg_tr,  H, use_log=False), index=valid_months)
        pt_sg  = pd.Series(forecast_holt_trend(total_sg_tr, H, use_log=True),  index=valid_months)
        pf_ns  = pd.Series(forecast_holt_trend(freq_ns_tr,  H, use_log=False), index=valid_months)
        pt_ns  = pd.Series(forecast_holt_trend(total_ns_tr, H, use_log=True),  index=valid_months)

    elif family == "recent_mean":
        pf_sg  = pd.Series(forecast_recent_mean(freq_sg_tr,  H, window=3), index=valid_months)
        pt_sg  = pd.Series(forecast_recent_mean(total_sg_tr, H, window=3), index=valid_months)
        pf_ns  = pd.Series(forecast_recent_mean(freq_ns_tr,  H, window=3), index=valid_months)
        pt_ns  = pd.Series(forecast_recent_mean(total_ns_tr, H, window=3), index=valid_months)

    else:
        raise ValueError("Unknown family: " + str(family))

    # portfolio
    freq_hat  = (pf_sg + pf_ns).astype(float)
    total_hat = (pt_sg + pt_ns).astype(float)
    sev_hat   = total_hat / np.maximum(freq_hat, EPS)

    # output dataframe pred
    pred = pd.DataFrame({
        "year_month": valid_months,
        "pred_freq": freq_hat.values,
        "pred_total": total_hat.values,
        "pred_sev": sev_hat.values,
    })
    return pred

def truth_portfolio(panel: pd.DataFrame, valid_months: list) -> pd.DataFrame:
    tmp = panel.set_index("year_month").loc[valid_months].copy()
    freq = tmp["frequency_sg"] + tmp["frequency_nonsg"]
    total = tmp["total_claim_sg"] + tmp["total_claim_nonsg"]
    sev = total / np.maximum(freq, EPS)
    return pd.DataFrame({
        "year_month": valid_months,
        "true_freq": freq.values.astype(float),
        "true_total": total.values.astype(float),
        "true_sev": sev.values.astype(float),
    })

def score_split(truth_df: pd.DataFrame, pred_df: pd.DataFrame) -> dict:
    # align
    t = truth_df.set_index("year_month")
    p = pred_df.set_index("year_month")
    idx = t.index.intersection(p.index)

    mf = mape(t.loc[idx, "true_freq"],  p.loc[idx, "pred_freq"])
    ms = mape(t.loc[idx, "true_sev"],   p.loc[idx, "pred_sev"])
    mt = mape(t.loc[idx, "true_total"], p.loc[idx, "pred_total"])
    return {"MAPE_freq": mf, "MAPE_sev": ms, "MAPE_total": mt, "FINAL_SCORE": (mf+ms+mt)/3.0}

# -------------------------
# Input: STAGE5_PANEL_WIDE harus ada
# -------------------------
if "STAGE5_PANEL_WIDE" not in globals():
    raise ValueError("STAGE5_PANEL_WIDE tidak ditemukan. Jalankan STAGE 5 dulu.")

panel = STAGE5_PANEL_WIDE.copy().sort_values("year_month").reset_index(drop=True)

need_cols = ["year_month", "frequency_sg", "total_claim_sg", "frequency_nonsg", "total_claim_nonsg"]
miss = [c for c in need_cols if c not in panel.columns]
if miss:
    raise ValueError(f"Kolom panel belum lengkap: {miss}")

months_ym = panel["year_month"].tolist()
splits = make_rolling_splits(months_ym, horizon=5)

# Bias: biasanya yang paling relevan adalah split terakhir (mendekati test),
# tapi kita tetap hitung semua split yang valid untuk gambaran stabilitas.
print(f"[STAGE7] Total possible rolling splits (h=5): {len(splits)}")
print("[STAGE7] Last month in history:", months_ym[-1])

# -------------------------
# Run backtest for each family
# -------------------------
families = ["yoy_anchor", "holt_trend", "recent_mean"]
rows = []

for si, (tr, va) in enumerate(splits):
    train_months = [period_to_ym(p) for p in pd.period_range(tr[0], tr[1], freq="M")]
    valid_months = [period_to_ym(p) for p in pd.period_range(va[0], va[1], freq="M")]

    truth_df = truth_portfolio(panel, valid_months)

    for fam in families:
        pred_df = predict_family_for_split(panel, train_months, valid_months, fam)
        sc = score_split(truth_df, pred_df)
        rows.append({
            "split": si,
            "train_start": period_to_ym(tr[0]),
            "train_end": period_to_ym(tr[1]),
            "valid_start": period_to_ym(va[0]),
            "valid_end": period_to_ym(va[1]),
            "family": fam,
            **sc
        })

bt = pd.DataFrame(rows)

# -------------------------
# Summary: mean + worst split (robust)
# -------------------------
summary = (bt.groupby("family")
             .agg(
                 mean_FINAL=("FINAL_SCORE","mean"),
                 worst_FINAL=("FINAL_SCORE","max"),
                 mean_mape_freq=("MAPE_freq","mean"),
                 mean_mape_sev=("MAPE_sev","mean"),
                 mean_mape_total=("MAPE_total","mean"),
                 n_splits=("split","nunique")
             )
             .sort_values("mean_FINAL")
             .reset_index()
          )

print("\n[STAGE7] Backtest summary (lower is better):")
display(summary)

print("\n[STAGE7] Detailed results (last 6 rows):")
display(bt.sort_values(["split","family"]).tail(6))

# Simpan artifact untuk tahap berikutnya (ensemble + tuning)
STAGE7_BACKTEST_RESULTS = bt.copy()
STAGE7_BACKTEST_SUMMARY = summary.copy()

print("[STAGE7] Stored: STAGE7_BACKTEST_RESULTS, STAGE7_BACKTEST_SUMMARY")

[STAGE7] Total possible rolling splits (h=5): 14
[STAGE7] Last month in history: 2025-07

[STAGE7] Backtest summary (lower is better):


,family,mean_FINAL,worst_FINAL,mean_mape_freq,mean_mape_sev,mean_mape_total,n_splits
0,recent_mean,0.145707,0.378163,0.102511,0.154915,0.179696,14
1,yoy_anchor,0.173642,0.378163,0.146880,0.153092,0.220953,14
2,holt_trend,0.174168,0.378163,0.094184,0.188213,0.240107,14



[STAGE7] Detailed results (last 6 rows):


,split,train_start,train_end,valid_start,valid_end,family,MAPE_freq,MAPE_sev,MAPE_total,FINAL_SCORE
37,12,2024-01,2025-01,2025-02,2025-06,holt_trend,0.043394,0.162735,0.150654,0.118928
38,12,2024-01,2025-01,2025-02,2025-06,recent_mean,0.053921,0.154525,0.134927,0.114457
36,12,2024-01,2025-01,2025-02,2025-06,yoy_anchor,0.130217,0.090860,0.074542,0.098540
40,13,2024-01,2025-02,2025-03,2025-07,holt_trend,0.059008,0.069526,0.089209,0.072581
41,13,2024-01,2025-02,2025-03,2025-07,recent_mean,0.055802,0.058513,0.070907,0.061741
39,13,2024-01,2025-02,2025-03,2025-07,yoy_anchor,0.104626,0.102915,0.051674,0.086405


[STAGE7] Stored: STAGE7_BACKTEST_RESULTS, STAGE7_BACKTEST_SUMMARY


# Ensemble Weight Search (Optimize for Kaggle Score)

In [9]:
# =========================
# STAGE 8 — Ensemble Weight Search (Optimize for Kaggle Score)
# - Cari bobot ensemble untuk 3 model family: yoy_anchor, holt_trend, recent_mean
# - Bobot ditune untuk meminimalkan skor Kaggle-like = avg(MAPE_freq, MAPE_sev, MAPE_total)
# - Strategi stabil (tidak terlalu overfit):
#   * 1 set bobot untuk semua FREQ (dipakai ke SG & NONSG)
#   * 1 set bobot untuk semua TOTAL (dipakai ke SG & NONSG)
# - Evaluasi via rolling backtest horizon=5
# =========================

import numpy as np
import pandas as pd
from IPython.display import display

try:
    from statsmodels.tsa.holtwinters import ExponentialSmoothing
except Exception:
    # di Kaggle biasanya sudah ada. kalau belum:
    # !pip -q install statsmodels
    from statsmodels.tsa.holtwinters import ExponentialSmoothing

EPS = 1e-9

# -------------------------
# Basic utils
# -------------------------
def mape(y_true, y_pred, eps=EPS) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.maximum(np.abs(y_true), eps)
    return float(np.mean(np.abs((y_true - y_pred) / denom)))

def ym_to_period(ym: str) -> pd.Period:
    return pd.Period(ym, freq="M")

def period_to_ym(p: pd.Period) -> str:
    return p.strftime("%Y-%m")

def safe_nonneg(x):
    x = np.asarray(x, dtype=float)
    x = np.where(np.isfinite(x), x, 0.0)
    x = np.where(x < 0, 0.0, x)
    return x

def make_rolling_splits(months_ym: list, horizon=5):
    months = [ym_to_period(m) for m in months_ym]
    months = sorted(months)
    splits = []
    for i in range(len(months)):
        train_end = months[i]
        valid_start = train_end + 1
        valid_end = train_end + horizon
        if valid_end <= months[-1]:
            splits.append(((months[0], train_end), (valid_start, valid_end)))
    return splits

# -------------------------
# Forecast models (family)
# -------------------------
def forecast_yoy_anchor(y: pd.Series, future_months: list, growth_clip=(0.6, 1.5)) -> np.ndarray:
    idx = [ym_to_period(m) for m in y.index]
    yP = pd.Series(y.values, index=idx).sort_index()

    ratios = []
    for p in yP.index:
        p_prev = p - 12
        if p_prev in yP.index and yP.loc[p_prev] > 0:
            ratios.append(yP.loc[p] / yP.loc[p_prev])
    g = float(np.median(ratios)) if len(ratios) >= 3 else 1.0
    g = float(np.clip(g, growth_clip[0], growth_clip[1]))

    tail = yP.iloc[-3:] if len(yP) >= 3 else yP
    fallback = float(np.median(tail.values)) if len(tail) else 0.0

    out = []
    for m in future_months:
        p = ym_to_period(m)
        ref = p - 12
        base = float(yP.loc[ref]) if ref in yP.index else fallback
        out.append(max(0.0, base * g))
    return safe_nonneg(out)

def forecast_holt_trend(y: pd.Series, horizon=5, use_log=False) -> np.ndarray:
    yy = y.astype(float).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    yy = pd.Series(safe_nonneg(yy.values), index=yy.index)

    if len(yy) < 4:
        base = float(np.mean(yy.values)) if len(yy) else 0.0
        return np.array([base]*horizon, dtype=float)

    if use_log:
        z = np.log1p(yy.values)
        fit = ExponentialSmoothing(
            z, trend="add", seasonal=None, damped_trend=True,
            initialization_method="estimated"
        ).fit(optimized=True)
        fc = np.asarray(fit.forecast(horizon), dtype=float)
        return safe_nonneg(np.expm1(fc))

    fit = ExponentialSmoothing(
        yy.values, trend="add", seasonal=None, damped_trend=True,
        initialization_method="estimated"
    ).fit(optimized=True)
    fc = np.asarray(fit.forecast(horizon), dtype=float)
    return safe_nonneg(fc)

def forecast_recent_mean(y: pd.Series, horizon=5, window=3) -> np.ndarray:
    yy = y.astype(float).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    yy = pd.Series(safe_nonneg(yy.values), index=yy.index)
    if len(yy) == 0:
        return np.zeros(horizon, dtype=float)
    w = min(window, len(yy))
    base = float(np.mean(yy.iloc[-w:].values))
    return np.array([max(0.0, base)]*horizon, dtype=float)

# -------------------------
# Build cached predictions per split (biar grid search cepat)
# -------------------------
if "STAGE5_PANEL_WIDE" not in globals():
    raise ValueError("STAGE5_PANEL_WIDE tidak ditemukan. Jalankan STAGE 5 dulu.")

panel = STAGE5_PANEL_WIDE.copy().sort_values("year_month").reset_index(drop=True)

need_cols = ["year_month", "frequency_sg", "total_claim_sg", "frequency_nonsg", "total_claim_nonsg"]
miss = [c for c in need_cols if c not in panel.columns]
if miss:
    raise ValueError(f"Kolom panel belum lengkap: {miss}")

months_all = panel["year_month"].tolist()
splits = make_rolling_splits(months_all, horizon=5)
if len(splits) == 0:
    raise ValueError("Split backtest tidak terbentuk. Pastikan historis bulan cukup untuk horizon=5.")

# fokus stabil: ambil 3 split terakhir (paling mirip kondisi test)
# kalau split < 3, pakai semua split
if len(splits) >= 3:
    eval_splits = list(range(len(splits)-3, len(splits)))
    split_weights = np.array([0.25, 0.30, 0.45], dtype=float)  # lebih berat ke split paling akhir
else:
    eval_splits = list(range(len(splits)))
    split_weights = np.ones(len(eval_splits), dtype=float) / max(1, len(eval_splits))

families = ["yoy_anchor", "holt_trend", "recent_mean"]
series_cols = {
    "freq_sg": "frequency_sg",
    "total_sg": "total_claim_sg",
    "freq_nonsg": "frequency_nonsg",
    "total_nonsg": "total_claim_nonsg",
}

# cache: cache_preds[si][series_key][family] = np.array length H
cache_preds = {}
# cache truth portfolio per split
cache_truth = {}

for si in eval_splits:
    (tr, va) = splits[si]
    train_months = [period_to_ym(p) for p in pd.period_range(tr[0], tr[1], freq="M")]
    valid_months = [period_to_ym(p) for p in pd.period_range(va[0], va[1], freq="M")]
    H = len(valid_months)

    # truth portfolio
    tmp = panel.set_index("year_month").loc[valid_months].copy()
    true_freq = (tmp["frequency_sg"] + tmp["frequency_nonsg"]).astype(float).values
    true_total = (tmp["total_claim_sg"] + tmp["total_claim_nonsg"]).astype(float).values
    true_sev = true_total / np.maximum(true_freq, EPS)
    cache_truth[si] = {
        "valid_months": valid_months,
        "true_freq": safe_nonneg(true_freq),
        "true_total": safe_nonneg(true_total),
        "true_sev": safe_nonneg(true_sev),
    }

    cache_preds[si] = {}
    for sk, col in series_cols.items():
        y_train = panel.set_index("year_month").loc[train_months, col].astype(float)
        y_train = y_train.replace([np.inf, -np.inf], np.nan).fillna(0.0)
        y_train = pd.Series(safe_nonneg(y_train.values), index=y_train.index)

        is_total = sk.startswith("total")

        cache_preds[si][sk] = {}
        cache_preds[si][sk]["yoy_anchor"] = forecast_yoy_anchor(y_train, valid_months)
        cache_preds[si][sk]["holt_trend"] = forecast_holt_trend(y_train, horizon=H, use_log=is_total)
        cache_preds[si][sk]["recent_mean"] = forecast_recent_mean(y_train, horizon=H, window=3)

print(f"[STAGE8] Cached predictions for {len(eval_splits)} splits (weighted), horizon=5.")
print("[STAGE8] Evaluated splits:", eval_splits, "| weights:", split_weights.tolist())

# -------------------------
# Evaluator: given weight sets -> weighted score across splits
# -------------------------
def blend3(a, b, c, w):
    # w = (w0,w1,w2), arrays same length
    return w[0]*a + w[1]*b + w[2]*c

def eval_weights(w_freq, w_total) -> dict:
    # w_freq and w_total are tuples/list length 3, sum=1, non-neg
    scores = []
    mf_list, ms_list, mt_list = [], [], []

    for wi, si in enumerate(eval_splits):
        w_split = split_weights[wi]
        truth = cache_truth[si]

        # predict per series
        # freq
        pf_sg = blend3(cache_preds[si]["freq_sg"]["yoy_anchor"],
                       cache_preds[si]["freq_sg"]["holt_trend"],
                       cache_preds[si]["freq_sg"]["recent_mean"], w_freq)
        pf_ns = blend3(cache_preds[si]["freq_nonsg"]["yoy_anchor"],
                       cache_preds[si]["freq_nonsg"]["holt_trend"],
                       cache_preds[si]["freq_nonsg"]["recent_mean"], w_freq)

        # total
        pt_sg = blend3(cache_preds[si]["total_sg"]["yoy_anchor"],
                       cache_preds[si]["total_sg"]["holt_trend"],
                       cache_preds[si]["total_sg"]["recent_mean"], w_total)
        pt_ns = blend3(cache_preds[si]["total_nonsg"]["yoy_anchor"],
                       cache_preds[si]["total_nonsg"]["holt_trend"],
                       cache_preds[si]["total_nonsg"]["recent_mean"], w_total)

        pred_freq = safe_nonneg(pf_sg + pf_ns)
        pred_total = safe_nonneg(pt_sg + pt_ns)
        pred_sev = pred_total / np.maximum(pred_freq, EPS)

        mf = mape(truth["true_freq"], pred_freq)
        ms = mape(truth["true_sev"], pred_sev)
        mt = mape(truth["true_total"], pred_total)
        final = (mf + ms + mt)/3.0

        scores.append(w_split * final)
        mf_list.append(w_split * mf)
        ms_list.append(w_split * ms)
        mt_list.append(w_split * mt)

    return {
        "FINAL_SCORE": float(np.sum(scores)),
        "MAPE_freq": float(np.sum(mf_list)),
        "MAPE_sev": float(np.sum(ms_list)),
        "MAPE_total": float(np.sum(mt_list)),
    }

# -------------------------
# Grid generator for weights (sum=1)
# -------------------------
def weight_grid(step=0.1):
    ws = np.arange(0, 1 + 1e-9, step)
    out = []
    for w0 in ws:
        for w1 in ws:
            w2 = 1.0 - w0 - w1
            if w2 < -1e-9:
                continue
            if w2 < 0:
                w2 = 0.0
            # renorm kecil untuk floating
            s = w0 + w1 + w2
            if s <= 0:
                continue
            w = (w0/s, w1/s, w2/s)
            out.append(w)
    # dedup (floating safe)
    uniq = []
    seen = set()
    for w in out:
        key = tuple([round(x, 6) for x in w])
        if key not in seen:
            uniq.append(w)
            seen.add(key)
    return uniq

GRID_STEP = 0.1  # stabil + cepat; nanti bisa diperkecil (0.05) setelah dapat area terbaik
WGRID = weight_grid(step=GRID_STEP)

# -------------------------
# 1) Search w_freq with w_total fixed (baseline holt for total)
# -------------------------
w_total_base = (0.0, 1.0, 0.0)  # holt_trend
best_freq = None
best_score = 1e18
rows_freq = []

for w_freq in WGRID:
    sc = eval_weights(w_freq, w_total_base)
    rows_freq.append({"w_freq": w_freq, **sc})
    if sc["FINAL_SCORE"] < best_score:
        best_score = sc["FINAL_SCORE"]
        best_freq = w_freq

df_freq = pd.DataFrame(rows_freq).sort_values("FINAL_SCORE").reset_index(drop=True)

print("\n[STAGE8] Best w_freq (with w_total=holt_trend):", best_freq)
display(df_freq.head(10))

# -------------------------
# 2) Search w_total with w_freq fixed to best_freq
# -------------------------
best_total = None
best_score2 = 1e18
rows_total = []

for w_total in WGRID:
    sc = eval_weights(best_freq, w_total)
    rows_total.append({"w_total": w_total, **sc})
    if sc["FINAL_SCORE"] < best_score2:
        best_score2 = sc["FINAL_SCORE"]
        best_total = w_total

df_total = pd.DataFrame(rows_total).sort_values("FINAL_SCORE").reset_index(drop=True)

print("\n[STAGE8] Best w_total (with w_freq fixed):", best_total)
display(df_total.head(10))

# -------------------------
# 3) Optional refine around best (pakai step lebih kecil 0.05, tapi tetap aman)
# -------------------------
def refine_grid(center_w, step=0.05):
    # generate local weights around center_w using +/- step and re-normalize
    c = np.array(center_w, dtype=float)
    candidates = []
    deltas = [-step, 0.0, step]
    for d0 in deltas:
        for d1 in deltas:
            for d2 in deltas:
                w = c + np.array([d0, d1, d2], dtype=float)
                w = np.where(w < 0, 0.0, w)
                s = w.sum()
                if s <= 0:
                    continue
                w = (w / s).tolist()
                key = tuple([round(x, 6) for x in w])
                candidates.append((key, tuple(w)))
    # unique
    uniq = {}
    for k, w in candidates:
        uniq[k] = w
    return list(uniq.values())

REFINE_STEP = 0.05
local_freq = refine_grid(best_freq, step=REFINE_STEP)
local_total = refine_grid(best_total, step=REFINE_STEP)

best_pair = (best_freq, best_total)
best_sc = eval_weights(best_freq, best_total)["FINAL_SCORE"]

for wf in local_freq:
    sc = eval_weights(wf, best_total)["FINAL_SCORE"]
    if sc < best_sc:
        best_sc = sc
        best_pair = (wf, best_total)

for wt in local_total:
    sc = eval_weights(best_pair[0], wt)["FINAL_SCORE"]
    if sc < best_sc:
        best_sc = sc
        best_pair = (best_pair[0], wt)

best_freq_ref, best_total_ref = best_pair
best_metrics = eval_weights(best_freq_ref, best_total_ref)

print("\n[STAGE8] Refined best weights:")
print("  w_freq  =", best_freq_ref, "  (yoy_anchor, holt_trend, recent_mean)")
print("  w_total =", best_total_ref, " (yoy_anchor, holt_trend, recent_mean)")
print("[STAGE8] Weighted backtest score (lower better):")
print(best_metrics)

# -------------------------
# Store artifacts
# -------------------------
STAGE8_WEIGHTS = {
    "w_freq": best_freq_ref,
    "w_total": best_total_ref,
    "grid_step_global": GRID_STEP,
    "refine_step": REFINE_STEP,
    "eval_splits": eval_splits,
    "split_weights": split_weights.tolist(),
}
STAGE8_FREQ_TABLE = df_freq.copy()
STAGE8_TOTAL_TABLE = df_total.copy()
STAGE8_BEST_METRICS = best_metrics

print("\n[STAGE8] Stored: STAGE8_WEIGHTS, STAGE8_FREQ_TABLE, STAGE8_TOTAL_TABLE, STAGE8_BEST_METRICS")

[STAGE8] Cached predictions for 3 splits (weighted), horizon=5.
[STAGE8] Evaluated splits: [11, 12, 13] | weights: [0.25, 0.3, 0.45]

[STAGE8] Best w_freq (with w_total=holt_trend): (np.float64(0.0), np.float64(1.0), np.float64(0.0))


,w_freq,FINAL_SCORE,MAPE_freq,MAPE_sev,MAPE_total
0,"(0.0, 1.0, 0.0)",0.098864,0.056196,0.115521,0.124875
1,"(0.0, 0.9, 0.09999999999999998)",0.099704,0.057658,0.116577,0.124875
2,"(0.0, 0.8, 0.19999999999999996)",0.100536,0.059121,0.117611,0.124875
3,"(0.1, 0.9, 0.0)",0.101195,0.060306,0.118405,0.124875
4,"(0.0, 0.7000000000000001, 0.29999999999999993)",0.101392,0.060679,0.118622,0.124875
5,"(0.1, 0.8, 0.09999999999999998)",0.101964,0.061541,0.119476,0.124875
6,"(0.0, 0.6000000000000001, 0.3999999999999999)",0.102489,0.062980,0.119612,0.124875
7,"(0.1, 0.7000000000000001, 0.19999999999999996)",0.102829,0.063087,0.120525,0.124875
8,"(0.0, 0.5, 0.5)",0.103579,0.065280,0.120580,0.124875
9,"(0.1, 0.6000000000000001, 0.29999999999999993)",0.103762,0.064860,0.121551,0.124875



[STAGE8] Best w_total (with w_freq fixed): (np.float64(0.30000000000000004), np.float64(0.0), np.float64(0.7))


,w_total,FINAL_SCORE,MAPE_freq,MAPE_sev,MAPE_total
0,"(0.30000000000000004, 0.0, 0.7)",0.086289,0.056196,0.098108,0.104563
1,"(0.4, 0.0, 0.6)",0.086455,0.056196,0.100749,0.102421
2,"(0.4, 0.1, 0.5)",0.086584,0.056196,0.102296,0.101258
3,"(0.2, 0.0, 0.8)",0.086617,0.056196,0.095508,0.108149
4,"(0.30000000000000004, 0.1, 0.6)",0.086887,0.056196,0.099655,0.104812
5,"(0.5, 0.0, 0.5)",0.086988,0.056196,0.103391,0.101379
6,"(0.5, 0.1, 0.4)",0.087116,0.056196,0.104938,0.100216
7,"(0.4, 0.2, 0.39999999999999997)",0.087171,0.056196,0.103844,0.101475
8,"(0.2, 0.1, 0.7000000000000001)",0.087202,0.056196,0.097013,0.108398
9,"(0.5, 0.2, 0.3)",0.087245,0.056196,0.106485,0.099052



[STAGE8] Refined best weights:
  w_freq  = (np.float64(0.0), np.float64(1.0), np.float64(0.0))   (yoy_anchor, holt_trend, recent_mean)
  w_total = (0.33333333333333337, 0.0, 0.6666666666666666)  (yoy_anchor, holt_trend, recent_mean)
[STAGE8] Weighted backtest score (lower better):
{'FINAL_SCORE': 0.08618377848406795, 'MAPE_freq': 0.0561958474563647, 'MAPE_sev': 0.09898816926484755, 'MAPE_total': 0.1033673187309916}

[STAGE8] Stored: STAGE8_WEIGHTS, STAGE8_FREQ_TABLE, STAGE8_TOTAL_TABLE, STAGE8_BEST_METRICS


# Bias Calibration + Guardrails (MAPE Killer)

In [10]:
# =========================
# STAGE 9 — Bias Calibration + Guardrails (MAPE Killer)
# - Tujuan: turunkan MAPE dengan:
#   (A) Multiplicative bias calibration (median(y_true / y_pred)) -> anti-bias
#   (B) Corridor clipping (train-only quantiles) -> anti outlier forecast
#   (C) Shrink to baseline (seasonal/YoY) -> anti overfit & lebih stabil
#
# Output:
#   STAGE9_PARAMS  : semua parameter kalibrasi + guardrail
#   STAGE9_APPLY() : fungsi untuk menerapkan ke prediksi final (future months)
# =========================

import numpy as np
import pandas as pd
from IPython.display import display

EPS = 1e-9

# -------------------------
# Requirements check
# -------------------------
if "STAGE5_PANEL_WIDE" not in globals():
    raise ValueError("STAGE5_PANEL_WIDE tidak ditemukan. Jalankan STAGE 5 dulu.")
if "STAGE8_WEIGHTS" not in globals():
    raise ValueError("STAGE8_WEIGHTS tidak ditemukan. Jalankan STAGE 8 dulu (ensemble weight search).")

panel = STAGE5_PANEL_WIDE.copy().sort_values("year_month").reset_index(drop=True)

need_cols = ["year_month", "frequency_sg", "total_claim_sg", "frequency_nonsg", "total_claim_nonsg"]
miss = [c for c in need_cols if c not in panel.columns]
if miss:
    raise ValueError(f"Kolom panel belum lengkap: {miss}")

w_freq  = tuple(STAGE8_WEIGHTS["w_freq"])
w_total = tuple(STAGE8_WEIGHTS["w_total"])

# -------------------------
# Shared helpers
# -------------------------
def ym_to_period(ym: str) -> pd.Period:
    return pd.Period(ym, freq="M")

def period_to_ym(p: pd.Period) -> str:
    return p.strftime("%Y-%m")

def make_rolling_splits(months_ym: list, horizon=5):
    months = [ym_to_period(m) for m in months_ym]
    months = sorted(months)
    splits = []
    for i in range(len(months)):
        train_end = months[i]
        valid_start = train_end + 1
        valid_end = train_end + horizon
        if valid_end <= months[-1]:
            splits.append(((months[0], train_end), (valid_start, valid_end)))
    return splits

def safe_nonneg(x):
    x = np.asarray(x, dtype=float)
    x = np.where(np.isfinite(x), x, 0.0)
    x = np.where(x < 0, 0.0, x)
    return x

def mape(y_true, y_pred, eps=EPS) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.maximum(np.abs(y_true), eps)
    return float(np.mean(np.abs((y_true - y_pred) / denom)))

def blend3(a, b, c, w):
    return w[0]*a + w[1]*b + w[2]*c

# --- model families (harus sama seperti STAGE 8) ---
try:
    from statsmodels.tsa.holtwinters import ExponentialSmoothing
except Exception:
    from statsmodels.tsa.holtwinters import ExponentialSmoothing

def forecast_yoy_anchor(y: pd.Series, future_months: list, growth_clip=(0.6, 1.5)) -> np.ndarray:
    idx = [ym_to_period(m) for m in y.index]
    yP = pd.Series(y.values, index=idx).sort_index()

    ratios = []
    for p in yP.index:
        p_prev = p - 12
        if p_prev in yP.index and yP.loc[p_prev] > 0:
            ratios.append(yP.loc[p] / yP.loc[p_prev])
    g = float(np.median(ratios)) if len(ratios) >= 3 else 1.0
    g = float(np.clip(g, growth_clip[0], growth_clip[1]))

    tail = yP.iloc[-3:] if len(yP) >= 3 else yP
    fallback = float(np.median(tail.values)) if len(tail) else 0.0

    out = []
    for m in future_months:
        p = ym_to_period(m)
        ref = p - 12
        base = float(yP.loc[ref]) if ref in yP.index else fallback
        out.append(max(0.0, base * g))
    return safe_nonneg(out)

def forecast_holt_trend(y: pd.Series, horizon=5, use_log=False) -> np.ndarray:
    yy = y.astype(float).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    yy = pd.Series(safe_nonneg(yy.values), index=yy.index)

    if len(yy) < 4:
        base = float(np.mean(yy.values)) if len(yy) else 0.0
        return np.array([base]*horizon, dtype=float)

    if use_log:
        z = np.log1p(yy.values)
        fit = ExponentialSmoothing(
            z, trend="add", seasonal=None, damped_trend=True,
            initialization_method="estimated"
        ).fit(optimized=True)
        fc = np.asarray(fit.forecast(horizon), dtype=float)
        return safe_nonneg(np.expm1(fc))

    fit = ExponentialSmoothing(
        yy.values, trend="add", seasonal=None, damped_trend=True,
        initialization_method="estimated"
    ).fit(optimized=True)
    fc = np.asarray(fit.forecast(horizon), dtype=float)
    return safe_nonneg(fc)

def forecast_recent_mean(y: pd.Series, horizon=5, window=3) -> np.ndarray:
    yy = y.astype(float).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    yy = pd.Series(safe_nonneg(yy.values), index=yy.index)
    if len(yy) == 0:
        return np.zeros(horizon, dtype=float)
    w = min(window, len(yy))
    base = float(np.mean(yy.iloc[-w:].values))
    return np.array([max(0.0, base)]*horizon, dtype=float)

def get_train_series(panel, train_months, col):
    s = panel.set_index("year_month").loc[train_months, col].astype(float)
    s = s.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return pd.Series(safe_nonneg(s.values), index=s.index)

def predict_ensemble_series(panel, train_months, valid_months, col, is_total, w):
    y_tr = get_train_series(panel, train_months, col)
    H = len(valid_months)
    a = forecast_yoy_anchor(y_tr, valid_months)
    b = forecast_holt_trend(y_tr, horizon=H, use_log=is_total)
    c = forecast_recent_mean(y_tr, horizon=H, window=3)
    return blend3(a, b, c, w)

# -------------------------
# (A) Bias calibration from rolling backtest (train-only)
# Kita hitung ratio median(y_true / y_pred) untuk:
# - portfolio freq
# - portfolio total
# Tidak kita kalibrasi severity langsung, karena severity akan di-derive dari total/freq.
# -------------------------
months_all = panel["year_month"].tolist()
splits = make_rolling_splits(months_all, horizon=5)

# pakai split terakhir (paling mirip test) lebih berat.
# kalau ada STAGE8_WEIGHTS split_weights, pakai itu.
if "eval_splits" in STAGE8_WEIGHTS and "split_weights" in STAGE8_WEIGHTS:
    eval_splits = STAGE8_WEIGHTS["eval_splits"]
    split_weights = np.array(STAGE8_WEIGHTS["split_weights"], dtype=float)
else:
    # fallback: last 3
    eval_splits = list(range(max(0, len(splits)-3), len(splits)))
    split_weights = np.array([0.25, 0.30, 0.45], dtype=float)[:len(eval_splits)]
    split_weights = split_weights / split_weights.sum()

ratios_freq = []
ratios_total = []

for wi, si in enumerate(eval_splits):
    (tr, va) = splits[si]
    train_months = [period_to_ym(p) for p in pd.period_range(tr[0], tr[1], freq="M")]
    valid_months = [period_to_ym(p) for p in pd.period_range(va[0], va[1], freq="M")]
    H = len(valid_months)

    tmp = panel.set_index("year_month").loc[valid_months].copy()
    true_freq = safe_nonneg((tmp["frequency_sg"] + tmp["frequency_nonsg"]).values)
    true_total = safe_nonneg((tmp["total_claim_sg"] + tmp["total_claim_nonsg"]).values)

    # pred per grup (freq & total) pakai ensemble weights hasil STAGE 8
    pf_sg = predict_ensemble_series(panel, train_months, valid_months, "frequency_sg",  False, w_freq)
    pf_ns = predict_ensemble_series(panel, train_months, valid_months, "frequency_nonsg",False, w_freq)
    pt_sg = predict_ensemble_series(panel, train_months, valid_months, "total_claim_sg", True,  w_total)
    pt_ns = predict_ensemble_series(panel, train_months, valid_months, "total_claim_nonsg",True, w_total)

    pred_freq = safe_nonneg(pf_sg + pf_ns)
    pred_total = safe_nonneg(pt_sg + pt_ns)

    # ratio = y_true / y_pred (multiplicative bias), handle pred=0
    r_f = true_freq / np.maximum(pred_freq, EPS)
    r_t = true_total / np.maximum(pred_total, EPS)

    # robust: clip ratio untuk hindari split yang aneh banget
    r_f = np.clip(r_f, 0.5, 2.0)
    r_t = np.clip(r_t, 0.5, 2.0)

    ratios_freq.append((wi, float(np.median(r_f))))
    ratios_total.append((wi, float(np.median(r_t))))

# weighted geometric-ish combine (pakai log-weighted median approach sederhana)
# Karena rasio multiplicative, combine lebih stabil di log-space
def combine_ratios(ratios_list, weights):
    vals = np.array([r for _, r in ratios_list], dtype=float)
    w = np.array(weights, dtype=float)
    w = w / w.sum()
    # log-mean weighted
    return float(np.exp(np.sum(w * np.log(np.maximum(vals, EPS)))))

bias_freq = combine_ratios(ratios_freq, split_weights)
bias_total = combine_ratios(ratios_total, split_weights)

print("[STAGE9] Bias factors (multiplicative):")
print("  bias_freq  =", bias_freq)
print("  bias_total =", bias_total)

# -------------------------
# (B) Corridor / guardrail quantiles (TRAIN-only idealnya).
# Untuk implementasi final, kita simpan corridor global dari historis sebagai fallback.
# (Nanti di STAGE 10, corridor bisa dibuat train-only; di sini kita siapkan default.)
# -------------------------
port_freq = (panel["frequency_sg"] + panel["frequency_nonsg"]).astype(float)
port_total = (panel["total_claim_sg"] + panel["total_claim_nonsg"]).astype(float)
port_sev = port_total / np.maximum(port_freq, EPS)

def qcorr(x, ql=0.05, qh=0.95):
    x = pd.to_numeric(x, errors="coerce").replace([np.inf,-np.inf], np.nan).dropna()
    if len(x) == 0:
        return (0.0, 0.0)
    return (float(x.quantile(ql)), float(x.quantile(qh)))

corr_freq  = qcorr(port_freq,  0.05, 0.95)
corr_total = qcorr(port_total, 0.05, 0.95)
corr_sev   = qcorr(port_sev,   0.05, 0.95)

print("[STAGE9] Default corridors (q05..q95, from full history):")
print("  freq :", corr_freq)
print("  total:", corr_total)
print("  sev  :", corr_sev)

# -------------------------
# (C) Shrink to baseline (YoY Anchor portfolio) — anti overfit
# - kita pakai alpha_freq dan alpha_total (0..1)
# - alpha akan ditune di backtest kecil (grid)
# -------------------------
def portfolio_yoy_baseline(panel, train_months, valid_months):
    # baseline portfolio = sum baseline per grup (yoy)
    y_fs = get_train_series(panel, train_months, "frequency_sg")
    y_fn = get_train_series(panel, train_months, "frequency_nonsg")
    y_ts = get_train_series(panel, train_months, "total_claim_sg")
    y_tn = get_train_series(panel, train_months, "total_claim_nonsg")

    base_freq = forecast_yoy_anchor(y_fs, valid_months) + forecast_yoy_anchor(y_fn, valid_months)
    base_total = forecast_yoy_anchor(y_ts, valid_months) + forecast_yoy_anchor(y_tn, valid_months)
    return safe_nonneg(base_freq), safe_nonneg(base_total)

def eval_alpha(alpha_freq, alpha_total):
    # evaluasi shrink pada eval_splits yang sama
    finals = []
    mf_w, ms_w, mt_w = [], [], []

    for wi, si in enumerate(eval_splits):
        (tr, va) = splits[si]
        train_months = [period_to_ym(p) for p in pd.period_range(tr[0], tr[1], freq="M")]
        valid_months = [period_to_ym(p) for p in pd.period_range(va[0], va[1], freq="M")]

        tmp = panel.set_index("year_month").loc[valid_months].copy()
        true_freq = safe_nonneg((tmp["frequency_sg"] + tmp["frequency_nonsg"]).values)
        true_total = safe_nonneg((tmp["total_claim_sg"] + tmp["total_claim_nonsg"]).values)
        true_sev = true_total / np.maximum(true_freq, EPS)

        # raw ensemble pred portfolio
        pf_sg = predict_ensemble_series(panel, train_months, valid_months, "frequency_sg",  False, w_freq)
        pf_ns = predict_ensemble_series(panel, train_months, valid_months, "frequency_nonsg",False, w_freq)
        pt_sg = predict_ensemble_series(panel, train_months, valid_months, "total_claim_sg", True,  w_total)
        pt_ns = predict_ensemble_series(panel, train_months, valid_months, "total_claim_nonsg",True, w_total)

        pred_freq = safe_nonneg(pf_sg + pf_ns) * bias_freq
        pred_total = safe_nonneg(pt_sg + pt_ns) * bias_total

        # baseline
        base_freq, base_total = portfolio_yoy_baseline(panel, train_months, valid_months)

        # shrink
        pred_freq = alpha_freq*pred_freq + (1-alpha_freq)*base_freq
        pred_total = alpha_total*pred_total + (1-alpha_total)*base_total

        # corridor clip
        pred_freq = np.clip(pred_freq, corr_freq[0], corr_freq[1] if corr_freq[1] > 0 else np.inf)
        pred_total = np.clip(pred_total, corr_total[0], corr_total[1] if corr_total[1] > 0 else np.inf)

        pred_sev = pred_total / np.maximum(pred_freq, EPS)
        pred_sev = np.clip(pred_sev, corr_sev[0], corr_sev[1] if corr_sev[1] > 0 else np.inf)

        mf = mape(true_freq, pred_freq)
        ms = mape(true_sev, pred_sev)
        mt = mape(true_total, pred_total)
        final = (mf+ms+mt)/3.0

        w = split_weights[wi]
        finals.append(w*final)
        mf_w.append(w*mf); ms_w.append(w*ms); mt_w.append(w*mt)

    return {
        "FINAL_SCORE": float(np.sum(finals)),
        "MAPE_freq": float(np.sum(mf_w)),
        "MAPE_sev": float(np.sum(ms_w)),
        "MAPE_total": float(np.sum(mt_w)),
        "alpha_freq": alpha_freq,
        "alpha_total": alpha_total,
    }

# grid kecil untuk alpha (cukup cepat)
alphas = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
rows = []
best = None
best_score = 1e18

for af in alphas:
    for at in alphas:
        sc = eval_alpha(af, at)
        rows.append(sc)
        if sc["FINAL_SCORE"] < best_score:
            best_score = sc["FINAL_SCORE"]
            best = sc

alpha_table = pd.DataFrame(rows).sort_values("FINAL_SCORE").reset_index(drop=True)

print("\n[STAGE9] Best shrink alphas:")
print(best)
display(alpha_table.head(10))

# -------------------------
# Build final STAGE9 params + apply function for future forecast
# -------------------------
STAGE9_PARAMS = {
    "bias_freq": bias_freq,
    "bias_total": bias_total,
    "corr_freq": corr_freq,
    "corr_total": corr_total,
    "corr_sev": corr_sev,
    "alpha_freq": float(best["alpha_freq"]),
    "alpha_total": float(best["alpha_total"]),
    "w_freq": w_freq,
    "w_total": w_total,
    "eval_splits": eval_splits,
    "split_weights": split_weights.tolist(),
}

def STAGE9_APPLY(panel_wide: pd.DataFrame, future_months: list) -> pd.DataFrame:
    """
    Terapkan ensemble + bias + shrink + corridor untuk FUTURE months.
    Output: pred portfolio (freq, total, sev)
    """
    panel_wide = panel_wide.copy().sort_values("year_month").reset_index(drop=True)
    last_ym = panel_wide["year_month"].iloc[-1]
    hist_months = panel_wide["year_month"].tolist()

    # full train pakai semua historis
    train_months = hist_months

    # raw ensemble per grup
    pf_sg = predict_ensemble_series(panel_wide, train_months, future_months, "frequency_sg",  False, STAGE9_PARAMS["w_freq"])
    pf_ns = predict_ensemble_series(panel_wide, train_months, future_months, "frequency_nonsg",False, STAGE9_PARAMS["w_freq"])
    pt_sg = predict_ensemble_series(panel_wide, train_months, future_months, "total_claim_sg", True,  STAGE9_PARAMS["w_total"])
    pt_ns = predict_ensemble_series(panel_wide, train_months, future_months, "total_claim_nonsg",True, STAGE9_PARAMS["w_total"])

    pred_freq = safe_nonneg(pf_sg + pf_ns) * STAGE9_PARAMS["bias_freq"]
    pred_total = safe_nonneg(pt_sg + pt_ns) * STAGE9_PARAMS["bias_total"]

    # baseline YoY
    base_freq, base_total = portfolio_yoy_baseline(panel_wide, train_months, future_months)

    # shrink
    af = STAGE9_PARAMS["alpha_freq"]
    at = STAGE9_PARAMS["alpha_total"]
    pred_freq = af*pred_freq + (1-af)*base_freq
    pred_total = at*pred_total + (1-at)*base_total

    # corridor
    lo_f, hi_f = STAGE9_PARAMS["corr_freq"]
    lo_t, hi_t = STAGE9_PARAMS["corr_total"]
    lo_s, hi_s = STAGE9_PARAMS["corr_sev"]

    pred_freq = np.clip(pred_freq, lo_f, hi_f if hi_f > 0 else np.inf)
    pred_total = np.clip(pred_total, lo_t, hi_t if hi_t > 0 else np.inf)

    pred_sev = pred_total / np.maximum(pred_freq, EPS)
    pred_sev = np.clip(pred_sev, lo_s, hi_s if hi_s > 0 else np.inf)

    # enforce coherence (Total = Freq * Sev)
    pred_total = pred_freq * pred_sev

    out = pd.DataFrame({
        "year_month": future_months,
        "pred_freq": pred_freq,
        "pred_sev": pred_sev,
        "pred_total": pred_total
    })
    return out

print("\n[STAGE9] Stored: STAGE9_PARAMS and function STAGE9_APPLY(panel, future_months)")

[STAGE9] Bias factors (multiplicative):
  bias_freq  = 0.9838365715987331
  bias_total = 1.017053029159349
[STAGE9] Default corridors (q05..q95, from full history):
  freq : (208.0, 280.4)
  total: (11008861478.031, 17758584483.70231)
  sev  : (46102714.36698347, 67486319.13475524)

[STAGE9] Best shrink alphas:
{'FINAL_SCORE': 0.08450508650035991, 'MAPE_freq': 0.058642984878790744, 'MAPE_sev': 0.09052413197293815, 'MAPE_total': 0.10434814264935086, 'alpha_freq': 0.9, 'alpha_total': 0.9}


,FINAL_SCORE,MAPE_freq,MAPE_sev,MAPE_total,alpha_freq,alpha_total
0,0.084505,0.058643,0.090524,0.104348,0.9,0.9
1,0.084971,0.058643,0.091816,0.104455,0.9,0.8
2,0.085017,0.058643,0.093040,0.103367,0.9,0.3
3,0.085063,0.063671,0.087170,0.104348,0.8,0.9
4,0.085128,0.058643,0.092795,0.103947,0.9,0.4
5,0.085247,0.058643,0.092061,0.105036,0.9,0.7
6,0.085522,0.058643,0.092306,0.105617,0.9,0.6
7,0.085539,0.058643,0.092550,0.105423,0.9,0.5
8,0.085782,0.063671,0.089221,0.104455,0.8,0.8
9,0.086345,0.063671,0.091997,0.103367,0.8,0.3



[STAGE9] Stored: STAGE9_PARAMS and function STAGE9_APPLY(panel, future_months)


# Reconciliation & Submission Builder

In [11]:
# =========================
# STAGE 10 — Reconciliation & Submission Builder
# - Ambil future months dari sample_submission (pasti match Kaggle)
# - Generate pred pakai STAGE9_APPLY (ensemble + bias + shrink + corridor)
# - Map ke format submission (15 baris)
# - Simpan: /kaggle/working/submission.csv
# =========================

import re
import numpy as np
import pandas as pd
from IPython.display import display

BASE_PATH = "/kaggle/input/datasets/dimaspashaakrilian/dsc-itb/"
PATH_SUB  = BASE_PATH + "sample_submission.csv"

# -------------------------
# Safety checks
# -------------------------
if "STAGE5_PANEL_WIDE" not in globals():
    raise ValueError("STAGE5_PANEL_WIDE tidak ditemukan. Jalankan STAGE 5 dulu.")
if "STAGE9_APPLY" not in globals():
    raise ValueError("STAGE9_APPLY tidak ditemukan. Jalankan STAGE 9 dulu sampai fungsi STAGE9_APPLY terbentuk.")

panel = STAGE5_PANEL_WIDE.copy().sort_values("year_month").reset_index(drop=True)
sample_sub = pd.read_csv(PATH_SUB)

if "id" not in sample_sub.columns:
    raise ValueError("sample_submission.csv tidak punya kolom 'id'. Cek file kamu.")
if "value" not in sample_sub.columns:
    # beberapa format Kaggle pakai kolom lain; kita bikin 'value' kalau belum ada
    sample_sub["value"] = np.nan

# -------------------------
# Get future months from submission ids (YYYY_MM -> YYYY-MM)
# -------------------------
ids = sample_sub["id"].astype(str).tolist()
mm = []
for s in ids:
    m = re.search(r"(\d{4})_(\d{2})_", s)
    if m:
        mm.append(f"{m.group(1)}-{m.group(2)}")
future_months = sorted(set(mm), key=lambda x: (int(x.split("-")[0]), int(x.split("-")[1])))

print("[STAGE10] Future months inferred from sample_submission:", future_months)

# -------------------------
# Predict portfolio for future months
# -------------------------
pred_future = STAGE9_APPLY(panel, future_months)  # columns: year_month, pred_freq, pred_sev, pred_total
display(pred_future)

# -------------------------
# Build prediction map (id -> value)
# -------------------------
def ym_to_prefix(ym: str) -> str:
    # "YYYY-MM" -> "YYYY_MM"
    y, m = ym.split("-")
    return f"{y}_{m}"

pred_map = {}
for _, r in pred_future.iterrows():
    ym = r["year_month"]
    prefix = ym_to_prefix(ym)

    freq = float(r["pred_freq"])
    sev  = float(r["pred_sev"])
    tot  = float(r["pred_total"])

    # final defensive: finite + non-negative
    for name, v in [("freq", freq), ("sev", sev), ("tot", tot)]:
        if not np.isfinite(v) or v < 0:
            raise ValueError(f"[STAGE10] Pred {name} invalid at {ym}: {v}")

    # enforce exact coherence again
    tot = freq * sev

    pred_map[f"{prefix}_Claim_Frequency"] = freq
    pred_map[f"{prefix}_Claim_Severity"]  = sev
    pred_map[f"{prefix}_Total_Claim"]     = tot

# -------------------------
# Fill submission
# -------------------------
sample_sub["value"] = sample_sub["id"].map(pred_map)

missing_ids = sample_sub[sample_sub["value"].isna()]["id"].tolist()
if missing_ids:
    raise ValueError(f"[STAGE10] Ada id di sample_submission yang tidak terisi. Contoh: {missing_ids[:10]}")

# final sanitize
vals = sample_sub["value"].astype(float).values
vals = np.where(np.isfinite(vals), vals, 0.0)
vals = np.where(vals < 0, 0.0, vals)
sample_sub["value"] = vals

# -------------------------
# Save
# -------------------------
OUT_PATH = "/kaggle/working/submission.csv"
sample_sub.to_csv(OUT_PATH, index=False)

print("[STAGE10] Saved:", OUT_PATH)
display(sample_sub.head(10))

[STAGE10] Future months inferred from sample_submission: ['2025-08', '2025-09', '2025-10', '2025-11', '2025-12']


,year_month,pred_freq,pred_sev,pred_total
0,2025-08,228.966764,6.088455e+07,1.394054e+10
1,2025-09,227.172135,5.909206e+07,1.342407e+10
2,2025-10,233.106966,5.831554e+07,1.359376e+10
3,2025-11,232.884100,6.059664e+07,1.411199e+10
4,2025-12,229.640630,5.839046e+07,1.340882e+10


[STAGE10] Saved: /kaggle/working/submission.csv


,id,value
0,2025_08_Claim_Frequency,2.289668e+02
1,2025_08_Claim_Severity,6.088455e+07
2,2025_08_Total_Claim,1.394054e+10
3,2025_09_Claim_Frequency,2.271721e+02
4,2025_09_Claim_Severity,5.909206e+07
5,2025_09_Total_Claim,1.342407e+10
6,2025_10_Claim_Frequency,2.331070e+02
7,2025_10_Claim_Severity,5.831554e+07
8,2025_10_Total_Claim,1.359376e+10
9,2025_11_Claim_Frequency,2.328841e+02
